# DriftGuard — Final Evaluation, Export and Inference Package

This notebook performs the final frozen-pipeline stage.

It will:

1. Verify the final-test SHA-256 before loading the file.
2. Restore the exact preprocessing functions used during training.
3. Load all three trained model families.
4. Evaluate the frozen models on the repository-disjoint final test.
5. Apply the frozen deterministic-rule and safety-hybrid engine.
6. Apply the frozen drift-scoring engine.
7. Generate final field-, commit-, file-, and repository-level reports.
8. Export a backend-ready inference bundle.
9. Generate API schemas, examples, tests, requirements, a model card, and an
   integration guide.

The final test is used only for final reporting. It must not be used to change:

- Model weights
- Rule definitions
- Risk thresholds
- Drift bands
- Scoring coefficients
- Primary model selection

In [1]:
import ast
import gc
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import math
import os
import platform
import re
import shutil
import sys
import textwrap
import time
import warnings
import zipfile

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import nbformat
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


print("=" * 72)
print("DRIFTGUARD — FINAL EVALUATION AND DEPLOYMENT EXPORT")
print("=" * 72)

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CONFIGS_DIR = PROJECT_ROOT / "configs"
MODELS_DIR = PROJECT_ROOT / "models"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean"

FINAL_TEST_PATH = (
    CLEAN_DATA_DIR
    / "test_final_evaluation_only.csv.gz"
)

SEALED_TEST_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "repository_disjoint_evaluation"
    / "manifests"
    / "sealed_final_test_manifest.json"
)

PRIMARY_SELECTION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "repository_disjoint_evaluation"
    / "manifests"
    / "primary_model_selection_manifest.json"
)

HYBRID_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "hybrid_engine"
    / "manifests"
    / "hybrid_engine_manifest.json"
)

SCORING_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "drift_scoring"
    / "manifests"
    / "drift_scoring_manifest.json"
)

FINAL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "final_evaluation"
)

TABLES_DIR = FINAL_OUTPUT_DIR / "tables"
PREDICTIONS_DIR = FINAL_OUTPUT_DIR / "predictions"
ERRORS_DIR = FINAL_OUTPUT_DIR / "errors"
TIMELINES_DIR = FINAL_OUTPUT_DIR / "timelines"
FIGURES_DIR = FINAL_OUTPUT_DIR / "figures"
MANIFESTS_DIR = FINAL_OUTPUT_DIR / "manifests"

DEPLOYMENT_ROOT = (
    PROJECT_ROOT
    / "deployment"
    / "driftguard_backend_bundle"
)

DEPLOYMENT_PACKAGE_DIR = (
    DEPLOYMENT_ROOT
    / "driftguard_inference"
)

DEPLOYMENT_MODELS_DIR = DEPLOYMENT_ROOT / "models"
DEPLOYMENT_CONFIGS_DIR = DEPLOYMENT_ROOT / "configs"
DEPLOYMENT_EXAMPLES_DIR = DEPLOYMENT_ROOT / "examples"
DEPLOYMENT_TESTS_DIR = DEPLOYMENT_ROOT / "tests"
DEPLOYMENT_DOCS_DIR = DEPLOYMENT_ROOT / "docs"

for directory in [
    FINAL_OUTPUT_DIR,
    TABLES_DIR,
    PREDICTIONS_DIR,
    ERRORS_DIR,
    TIMELINES_DIR,
    FIGURES_DIR,
    MANIFESTS_DIR,
    DEPLOYMENT_ROOT,
    DEPLOYMENT_PACKAGE_DIR,
    DEPLOYMENT_MODELS_DIR,
    DEPLOYMENT_CONFIGS_DIR,
    DEPLOYMENT_EXAMPLES_DIR,
    DEPLOYMENT_TESTS_DIR,
    DEPLOYMENT_DOCS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

warnings.filterwarnings("default")

print("Python            :", sys.version.split()[0])
print("Scikit-learn     :", sklearn.__version__)
print("PyTorch          :", torch.__version__)
print("Transformers     :", transformers.__version__)
print("Project root     :", PROJECT_ROOT)
print("Final test       :", FINAL_TEST_PATH)
print("Deployment bundle:", DEPLOYMENT_ROOT)

DRIFTGUARD — FINAL EVALUATION AND DEPLOYMENT EXPORT
Python            : 3.11.15
Scikit-learn     : 1.9.0
PyTorch          : 2.11.0+cu128
Transformers     : 5.14.1
Project root     : C:\Users\Lenovo\Desktop\DriftGuard
Final test       : C:\Users\Lenovo\Desktop\DriftGuard\data\clean\test_final_evaluation_only.csv.gz
Deployment bundle: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle


In [2]:
required_manifest_paths = {
    "sealed test manifest":
        SEALED_TEST_MANIFEST_PATH,

    "primary model manifest":
        PRIMARY_SELECTION_MANIFEST_PATH,

    "hybrid manifest":
        HYBRID_MANIFEST_PATH,

    "scoring manifest":
        SCORING_MANIFEST_PATH,
}

for artifact_name, artifact_path in (
    required_manifest_paths.items()
):
    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Missing {artifact_name}:\n"
            f"{artifact_path}"
        )


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


sealed_test_manifest = load_json(
    SEALED_TEST_MANIFEST_PATH
)

primary_selection_manifest = load_json(
    PRIMARY_SELECTION_MANIFEST_PATH
)

hybrid_manifest = load_json(
    HYBRID_MANIFEST_PATH
)

scoring_manifest = load_json(
    SCORING_MANIFEST_PATH
)

HYBRID_SETTINGS_PATH = (
    CONFIGS_DIR
    / "hybrid_engine_settings.json"
)

SCORING_SETTINGS_PATH = (
    CONFIGS_DIR
    / "drift_scoring_settings.json"
)

STRUCTURED_SETTINGS_PATH = (
    CONFIGS_DIR
    / "structured_model_training_settings.json"
)

for settings_path in [
    HYBRID_SETTINGS_PATH,
    SCORING_SETTINGS_PATH,
    STRUCTURED_SETTINGS_PATH,
]:
    if not settings_path.exists():
        raise FileNotFoundError(
            f"Missing frozen settings:\n{settings_path}"
        )

HYBRID_SETTINGS = load_json(
    HYBRID_SETTINGS_PATH
)

SCORING_SETTINGS = load_json(
    SCORING_SETTINGS_PATH
)

STRUCTURED_SETTINGS = load_json(
    STRUCTURED_SETTINGS_PATH
)

CLASS_ORDER = HYBRID_SETTINGS[
    "class_order"
]

RISK_RANK = HYBRID_SETTINGS[
    "risk_rank"
]

RANK_TO_CLASS = {
    int(rank): class_name
    for class_name, rank
    in RISK_RANK.items()
}

MODEL_WEIGHTS = HYBRID_SETTINGS[
    "model_weights"
]

SEVERITY_ANCHORS = SCORING_SETTINGS[
    "severity_anchors"
]

BAND_TO_LABEL = SCORING_SETTINGS[
    "band_to_label"
]

PRIMARY_MODEL_NAME = (
    primary_selection_manifest[
        "selected_primary_model"
    ]
)

PRODUCTION_VARIANT = (
    hybrid_manifest[
        "production_variant"
    ]
)

print("Primary learned model:", PRIMARY_MODEL_NAME)
print("Production hybrid    :", PRODUCTION_VARIANT)
print("Class order          :", CLASS_ORDER)
print("Model weights        :", MODEL_WEIGHTS)
print("Test status          :", sealed_test_manifest["status"])

Primary learned model: structured
Production hybrid    : safety_hybrid
Class order          : ['benign', 'low', 'medium', 'high', 'critical']
Model weights        : {'structured': 0.4, 'transformer': 0.35, 'text_baseline': 0.25}
Test status          : SEALED_NOT_LOADED


In [4]:
TEXT_MODEL_PATH = (
    MODELS_DIR
    / "baselines"
    / "best_baseline_model.joblib"
)

STRUCTURED_MODEL_PATH = (
    MODELS_DIR
    / "structured"
    / "best_structured_model.joblib"
)

TRANSFORMER_MODEL_PATH = (
    MODELS_DIR
    / "transformer"
    / "best_codeberta_model"
)

required_model_paths = {
    "text baseline":
        TEXT_MODEL_PATH,

    "structured model":
        STRUCTURED_MODEL_PATH,

    "transformer model":
        TRANSFORMER_MODEL_PATH,
}

for model_name, model_path in (
    required_model_paths.items()
):
    if not model_path.exists():
        raise FileNotFoundError(
            f"Missing {model_name}:\n"
            f"{model_path}"
        )

print("Frozen model artifacts located.")

for model_name, model_path in (
    required_model_paths.items()
):
    print(
        f"{model_name:<20}:",
        model_path,
    )

Frozen model artifacts located.
text baseline       : C:\Users\Lenovo\Desktop\DriftGuard\models\baselines\best_baseline_model.joblib
structured model    : C:\Users\Lenovo\Desktop\DriftGuard\models\structured\best_structured_model.joblib
transformer model   : C:\Users\Lenovo\Desktop\DriftGuard\models\transformer\best_codeberta_model


In [5]:
def calculate_file_sha256(
    file_path,
    chunk_size=1024 * 1024,
):
    sha256 = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


if not FINAL_TEST_PATH.exists():
    raise FileNotFoundError(
        f"Final test file is missing:\n"
        f"{FINAL_TEST_PATH}"
    )

expected_final_test_sha256 = (
    sealed_test_manifest[
        "sha256"
    ]
)

pre_access_final_test_sha256 = (
    calculate_file_sha256(
        FINAL_TEST_PATH
    )
)

print(
    "Expected SHA-256:",
    expected_final_test_sha256,
)

print(
    "Actual SHA-256  :",
    pre_access_final_test_sha256,
)

if (
    pre_access_final_test_sha256
    != expected_final_test_sha256
):
    raise ValueError(
        "The final-test SHA-256 does not match "
        "the sealed manifest."
    )

if (
    sealed_test_manifest[
        "status"
    ]
    != "SEALED_NOT_LOADED"
):
    raise ValueError(
        "The final-test manifest was not in the "
        "expected sealed state."
    )

print("Final-test integrity verification: PASSED")
print("The file has not yet been loaded.")

Expected SHA-256: cff0f85db740a195366fb81cbad26ba34859f40fe12bf8695b263d1cdedb9b67
Actual SHA-256  : cff0f85db740a195366fb81cbad26ba34859f40fe12bf8695b263d1cdedb9b67
Final-test integrity verification: PASSED
The file has not yet been loaded.


In [6]:
COMMON_NOTEBOOK_NAMESPACE = {
    "__builtins__": __builtins__,
    "os": os,
    "sys": sys,
    "json": json,
    "math": math,
    "re": re,
    "time": time,
    "hashlib": hashlib,
    "Path": Path,
    "Counter": Counter,
    "np": np,
    "pd": pd,
    "joblib": joblib,
    "torch": torch,
    "sklearn": sklearn,
    "transformers": transformers,
    "AutoTokenizer": AutoTokenizer,
    "AutoModelForSequenceClassification":
        AutoModelForSequenceClassification,
}


def restore_notebook_definitions(
    notebook_path,
):
    notebook_path = Path(
        notebook_path
    )

    if not notebook_path.exists():
        raise FileNotFoundError(
            f"Notebook not found:\n{notebook_path}"
        )

    notebook = nbformat.read(
        notebook_path,
        as_version=4,
    )

    namespace = dict(
        COMMON_NOTEBOOK_NAMESPACE
    )

    restored_functions = []
    restored_constants = []

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        try:
            syntax_tree = ast.parse(
                cell.source
            )
        except SyntaxError:
            continue

        for node in syntax_tree.body:
            should_execute = False
            restored_name = None

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            ):
                should_execute = True
                restored_name = node.name

            elif isinstance(
                node,
                ast.Assign,
            ):
                target_names = [
                    target.id
                    for target in node.targets
                    if isinstance(
                        target,
                        ast.Name,
                    )
                ]

                uppercase_targets = [
                    target_name
                    for target_name
                    in target_names
                    if target_name.isupper()
                ]

                if uppercase_targets:
                    should_execute = True
                    restored_name = ",".join(
                        uppercase_targets
                    )

            elif isinstance(
                node,
                ast.AnnAssign,
            ):
                if (
                    isinstance(
                        node.target,
                        ast.Name,
                    )
                    and node.target.id.isupper()
                ):
                    should_execute = True
                    restored_name = (
                        node.target.id
                    )

            if not should_execute:
                continue

            isolated_module = ast.Module(
                body=[node],
                type_ignores=[],
            )

            ast.fix_missing_locations(
                isolated_module
            )

            try:
                exec(
                    compile(
                        isolated_module,
                        str(
                            notebook_path
                        ),
                        "exec",
                    ),
                    namespace,
                )

                if isinstance(
                    node,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                        ast.ClassDef,
                    ),
                ):
                    restored_functions.append(
                        restored_name
                    )
                else:
                    restored_constants.append(
                        restored_name
                    )

            except Exception:
                # Some directory/path assignments depend on
                # notebook runtime variables and are unnecessary.
                continue

    return {
        "namespace": namespace,
        "functions": restored_functions,
        "constants": restored_constants,
    }


print("Notebook restoration helper loaded.")

Notebook restoration helper loaded.


In [9]:
notebook_07_path = (
    NOTEBOOKS_DIR
    / "07_train_baseline_models.ipynb"
)

notebook_08_path = (
    NOTEBOOKS_DIR
    / "08_train_structured_ml_models.ipynb"
)

notebook_09_path = (
    NOTEBOOKS_DIR
    / "09_train_nlp_transformer.ipynb"
)

notebook_11_path = (
    NOTEBOOKS_DIR
    / "11_hybrid_rules_model_engine.ipynb"
)

notebook_12_path = (
    NOTEBOOKS_DIR
    / "12_drift_scoring_engine.ipynb"
)

baseline_restoration = (
    restore_notebook_definitions(
        notebook_07_path
    )
)

structured_restoration = (
    restore_notebook_definitions(
        notebook_08_path
    )
)

transformer_restoration = (
    restore_notebook_definitions(
        notebook_09_path
    )
)

hybrid_restoration = (
    restore_notebook_definitions(
        notebook_11_path
    )
)

scoring_restoration = (
    restore_notebook_definitions(
        notebook_12_path
    )
)

baseline_namespace = baseline_restoration[
    "namespace"
]

structured_namespace = structured_restoration[
    "namespace"
]

transformer_namespace = transformer_restoration[
    "namespace"
]

hybrid_namespace = hybrid_restoration[
    "namespace"
]

scoring_namespace = scoring_restoration[
    "namespace"
]

structured_namespace[
    "STRUCTURED_SETTINGS"
] = STRUCTURED_SETTINGS

hybrid_namespace[
    "HYBRID_ENGINE_SETTINGS"
] = HYBRID_SETTINGS

hybrid_namespace[
    "CLASS_ORDER"
] = CLASS_ORDER

hybrid_namespace[
    "RISK_RANK"
] = RISK_RANK

hybrid_namespace[
    "RANK_TO_CLASS"
] = RANK_TO_CLASS

hybrid_namespace[
    "MODEL_WEIGHTS"
] = MODEL_WEIGHTS

scoring_namespace[
    "DRIFT_SCORING_SETTINGS"
] = SCORING_SETTINGS

scoring_namespace[
    "CLASS_ORDER"
] = CLASS_ORDER

scoring_namespace[
    "SEVERITY_ANCHORS"
] = SEVERITY_ANCHORS

scoring_namespace[
    "BAND_TO_LABEL"
] = BAND_TO_LABEL

required_structured_function = (
    "engineer_structured_features"
)

if (
    required_structured_function
    not in structured_namespace
):
    raise RuntimeError(
        "Could not restore "
        "engineer_structured_features."
    )

if (
    "apply_deterministic_rules_to_row"
    not in hybrid_namespace
):
    raise RuntimeError(
        "Could not restore the deterministic "
        "rule engine."
    )

if (
    "build_hybrid_decision"
    not in hybrid_namespace
):
    raise RuntimeError(
        "Could not restore the hybrid-decision "
        "function."
    )

if (
    "adjust_scores_to_decisions"
    not in hybrid_namespace
):
    raise RuntimeError(
        "Could not restore the score-adjustment "
        "function."
    )

for required_scoring_function in [
    "normalized_entropy",
    "assign_drift_band",
    "operation_is_removal",
    "operation_is_addition",
    "aggregate_event_scores",
    "calculate_decayed_pressure",
    "build_score_explanation",
]:
    if (
        required_scoring_function
        not in scoring_namespace
    ):
        raise RuntimeError(
            "Could not restore scoring function: "
            f"{required_scoring_function}"
        )

engineer_structured_features = (
    structured_namespace[
        "engineer_structured_features"
    ]
)

apply_deterministic_rules_to_row = (
    hybrid_namespace[
        "apply_deterministic_rules_to_row"
    ]
)

build_hybrid_decision = (
    hybrid_namespace[
        "build_hybrid_decision"
    ]
)

adjust_scores_to_decisions = (
    hybrid_namespace[
        "adjust_scores_to_decisions"
    ]
)

normalized_entropy = (
    scoring_namespace[
        "normalized_entropy"
    ]
)

assign_drift_band = (
    scoring_namespace[
        "assign_drift_band"
    ]
)

operation_is_removal = (
    scoring_namespace[
        "operation_is_removal"
    ]
)

operation_is_addition = (
    scoring_namespace[
        "operation_is_addition"
    ]
)

aggregate_event_scores = (
    scoring_namespace[
        "aggregate_event_scores"
    ]
)

calculate_decayed_pressure = (
    scoring_namespace[
        "calculate_decayed_pressure"
    ]
)

build_score_explanation = (
    scoring_namespace[
        "build_score_explanation"
    ]
)

print("Exact preprocessing and engine functions restored.")

Exact preprocessing and engine functions restored.


In [12]:
# Corrected Cell 8 — Restore text builders and their required settings

dummy_records = pd.DataFrame(
    [
        {
            "field_path":
                "spec.template.spec.containers[0].securityContext",

            "old_value":
                "runAsNonRoot: true",

            "new_value":
                "runAsNonRoot: false",

            "configuration_type":
                "yaml",

            "parser_mode":
                "structured",

            "operation":
                "modified",

            "file_path":
                "deployment.yaml",

            "commit_message":
                "update deployment settings",
        },

        {
            "field_path":
                "spec.resources.limits.memory",

            "old_value":
                "512Mi",

            "new_value":
                "",

            "configuration_type":
                "yaml",

            "parser_mode":
                "structured",

            "operation":
                "deleted",

            "file_path":
                "service.yaml",

            "commit_message":
                "remove obsolete limit",
        },
    ]
)


def discover_dataframe_text_builder(
    namespace,
    dataframe,
    preferred_terms=None,
    excluded_terms=None,
):
    preferred_terms = preferred_terms or []
    excluded_terms = excluded_terms or []

    candidates = []

    for function_name, function_object in namespace.items():
        if not callable(function_object):
            continue

        lowered_name = function_name.lower()

        if "text" not in lowered_name:
            continue

        if any(
            excluded_term in lowered_name
            for excluded_term in excluded_terms
        ):
            continue

        try:
            output = function_object(
                dataframe.copy()
            )
        except Exception:
            continue

        if isinstance(output, str):
            continue

        try:
            output_series = pd.Series(output)
        except Exception:
            continue

        if len(output_series) != len(dataframe):
            continue

        string_fraction = (
            output_series
            .fillna("")
            .map(
                lambda value:
                isinstance(value, str)
            )
            .mean()
        )

        if string_fraction < 0.75:
            continue

        score = 0

        for preferred_term in preferred_terms:
            if preferred_term in lowered_name:
                score += 50

        if "build" in lowered_name:
            score += 20

        if "input" in lowered_name:
            score += 10

        candidates.append(
            {
                "name": function_name,
                "function": function_object,
                "score": score,
            }
        )

    if not candidates:
        raise RuntimeError(
            "No compatible text-building function was found."
        )

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True,
    )

    return (
        candidates[0]["name"],
        candidates[0]["function"],
        candidates,
    )


# ---------------------------------------------------------------------
# Baseline text builder
# ---------------------------------------------------------------------

(
    baseline_text_builder_name,
    baseline_text_builder,
    baseline_builder_candidates,
) = discover_dataframe_text_builder(
    namespace=baseline_namespace,
    dataframe=dummy_records,
    preferred_terms=[
        "baseline",
        "combined",
        "document",
    ],
    excluded_terms=[
        "transformer",
    ],
)


# ---------------------------------------------------------------------
# Restore Transformer settings required by build_transformer_text()
# ---------------------------------------------------------------------

TRANSFORMER_SETTINGS_PATH = (
    CONFIGS_DIR
    / "transformer_training_settings.json"
)

if not TRANSFORMER_SETTINGS_PATH.exists():
    raise FileNotFoundError(
        "Missing Transformer settings file:\n"
        f"{TRANSFORMER_SETTINGS_PATH}"
    )

with TRANSFORMER_SETTINGS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    TRANSFORMER_SETTINGS = json.load(file)


# Insert required constants into the namespace in which the restored
# Transformer functions execute.

transformer_namespace.update(
    {
        "TRANSFORMER_SETTINGS":
            TRANSFORMER_SETTINGS,

        "CLASS_ORDER":
            TRANSFORMER_SETTINGS[
                "class_order"
            ],

        "MAXIMUM_SEQUENCE_LENGTH":
            TRANSFORMER_SETTINGS[
                "maximum_sequence_length"
            ],

        "MAXIMUM_FIELD_PATH_CHARACTERS":
            TRANSFORMER_SETTINGS[
                "maximum_field_path_characters"
            ],

        "MAXIMUM_OLD_VALUE_CHARACTERS":
            TRANSFORMER_SETTINGS[
                "maximum_old_value_characters"
            ],

        "MAXIMUM_NEW_VALUE_CHARACTERS":
            TRANSFORMER_SETTINGS[
                "maximum_new_value_characters"
            ],

        "MAXIMUM_FILE_PATH_CHARACTERS":
            TRANSFORMER_SETTINGS[
                "maximum_file_path_characters"
            ],

        "MAXIMUM_COMMIT_MESSAGE_CHARACTERS":
            TRANSFORMER_SETTINGS[
                "maximum_commit_message_characters"
            ],

        "np":
            np,

        "pd":
            pd,

        "re":
            re,

        "json":
            json,
    }
)


if (
    "build_transformer_text"
    not in transformer_namespace
):
    raise RuntimeError(
        "build_transformer_text was not restored "
        "from Notebook 09."
    )

transformer_text_builder_name = (
    "build_transformer_text"
)

transformer_text_builder = (
    transformer_namespace[
        transformer_text_builder_name
    ]
)


# The restored function already points to transformer_namespace, but
# explicitly update its globals for compatibility with dynamically
# compiled notebook functions.

transformer_text_builder.__globals__.update(
    transformer_namespace
)

transformer_text_builder.__globals__.update(
    {
        "TRANSFORMER_SETTINGS":
            TRANSFORMER_SETTINGS,

        "CLASS_ORDER":
            TRANSFORMER_SETTINGS[
                "class_order"
            ],

        "MAXIMUM_SEQUENCE_LENGTH":
            TRANSFORMER_SETTINGS[
                "maximum_sequence_length"
            ],
    }
)


# ---------------------------------------------------------------------
# Validate both builders
# ---------------------------------------------------------------------

baseline_dummy_output = pd.Series(
    baseline_text_builder(
        dummy_records.copy()
    )
).fillna("").astype(str)

transformer_dummy_output = pd.Series(
    transformer_text_builder(
        dummy_records.copy()
    )
).fillna("").astype(str)


if len(baseline_dummy_output) != len(dummy_records):
    raise ValueError(
        "Baseline text builder returned an "
        "incorrect number of records."
    )

if len(transformer_dummy_output) != len(dummy_records):
    raise ValueError(
        "Transformer text builder returned an "
        "incorrect number of records."
    )


print(
    "Baseline text builder   :",
    baseline_text_builder_name,
)

print(
    "Transformer text builder:",
    transformer_text_builder_name,
)

print(
    "Baseline test records   :",
    len(baseline_dummy_output),
)

print(
    "Transformer test records:",
    len(transformer_dummy_output),
)

print("\nSample Transformer input:")
print(transformer_dummy_output.iloc[0])

print("\nText-builder validation: PASSED")

Baseline text builder   : build_baseline_text
Transformer text builder: build_transformer_text
Baseline test records   : 2
Transformer test records: 2

Sample Transformer input:
[FIELD] spec.template.spec.containers[0].securityContext [OLD] runAsNonRoot: true [NEW] runAsNonRoot: false [CONFIG_TYPE] yaml [OPERATION] modified [PARSER] structured [FILE] deployment.yaml [COMMIT] update deployment settings

Text-builder validation: PASSED


In [13]:
def unwrap_saved_model(
    saved_object,
):
    if not isinstance(
        saved_object,
        dict,
    ):
        return saved_object

    preferred_keys = [
        "model",
        "pipeline",
        "best_model",
        "classifier",
        "estimator",
    ]

    for preferred_key in preferred_keys:
        if preferred_key in saved_object:
            return saved_object[
                preferred_key
            ]

    for value in saved_object.values():
        if (
            hasattr(
                value,
                "predict",
            )
            or hasattr(
                value,
                "forward",
            )
        ):
            return value

    raise ValueError(
        "No model object could be identified "
        "inside the saved dictionary."
    )


print("Loading text model...")

text_saved_object = joblib.load(
    TEXT_MODEL_PATH
)

text_baseline_model = unwrap_saved_model(
    text_saved_object
)

print(
    "Text model type:",
    type(
        text_baseline_model
    ).__name__,
)

print("\nLoading structured model...")

structured_saved_object = joblib.load(
    STRUCTURED_MODEL_PATH
)

structured_model = unwrap_saved_model(
    structured_saved_object
)

print(
    "Structured model type:",
    type(
        structured_model
    ).__name__,
)

print("\nLoading Transformer model...")

transformer_tokenizer = (
    AutoTokenizer.from_pretrained(
        TRANSFORMER_MODEL_PATH
    )
)

transformer_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        TRANSFORMER_MODEL_PATH
    )
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

transformer_model = transformer_model.to(
    DEVICE
)

transformer_model.eval()

print("Transformer device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

Loading text model...
Text model type: Pipeline

Loading structured model...
Structured model type: Pipeline

Loading Transformer model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Transformer device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [14]:
print("=" * 72)
print("UNSEALING FINAL TEST FOR FINAL EVALUATION")
print("=" * 72)

final_test_data = pd.read_csv(
    FINAL_TEST_PATH,
    compression="gzip",
    low_memory=False,
)

print(
    "Final-test records:",
    f"{len(final_test_data):,}",
)

if len(final_test_data) != 68_749:
    raise ValueError(
        "Unexpected final-test row count. "
        f"Expected 68,749 but found "
        f"{len(final_test_data):,}."
    )

label_candidates = [
    "training_target",
    "weak_label",
    "risk_label",
    "label",
    "evaluation_label",
]

final_test_label_column = None

for candidate in label_candidates:
    if candidate in final_test_data.columns:
        final_test_label_column = candidate
        break

if final_test_label_column is None:
    raise ValueError(
        "No final-test evaluation-label column "
        "could be identified."
    )

final_test_data[
    "evaluation_label"
] = (
    final_test_data[
        final_test_label_column
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

final_test_labeled_mask = (
    final_test_data[
        "evaluation_label"
    ]
    .isin(
        CLASS_ORDER
    )
)

print(
    "Evaluation-label source:",
    final_test_label_column,
)

print(
    "Labeled final-test records:",
    f"{final_test_labeled_mask.sum():,}",
)

print(
    "Unlabeled final-test records:",
    f"{(~final_test_labeled_mask).sum():,}",
)

UNSEALING FINAL TEST FOR FINAL EVALUATION
Final-test records: 68,749
Evaluation-label source: weak_label
Labeled final-test records: 61,899
Unlabeled final-test records: 6,850


In [15]:
reference_dataset_paths = [
    CLEAN_DATA_DIR
    / "train_temporal_model_train.csv.gz",

    CLEAN_DATA_DIR
    / "train_temporal_development.csv.gz",

    CLEAN_DATA_DIR
    / "validation_review_only.csv.gz",
]

reference_frames = []

for reference_path in reference_dataset_paths:
    if not reference_path.exists():
        raise FileNotFoundError(
            f"Missing reference dataset:\n"
            f"{reference_path}"
        )

    reference_frame = pd.read_csv(
        reference_path,
        compression="gzip",
        low_memory=False,
    )

    reference_frames.append(
        reference_frame
    )

reference_data = pd.concat(
    reference_frames,
    ignore_index=True,
)

reference_repositories = set(
    reference_data[
        "repository"
    ].dropna().astype(str)
)

test_repositories = set(
    final_test_data[
        "repository"
    ].dropna().astype(str)
)

repository_overlap = (
    reference_repositories
    & test_repositories
)

reference_commit_keys = set(
    zip(
        reference_data[
            "repository"
        ].astype(str),

        reference_data[
            "commit_hash"
        ].astype(str),
    )
)

test_commit_keys = set(
    zip(
        final_test_data[
            "repository"
        ].astype(str),

        final_test_data[
            "commit_hash"
        ].astype(str),
    )
)

commit_overlap = (
    reference_commit_keys
    & test_commit_keys
)

signature_column_candidates = [
    "change_signature_sha256",
    "change_signature",
]

signature_column = None

for candidate in signature_column_candidates:
    if (
        candidate
        in reference_data.columns
        and candidate
        in final_test_data.columns
    ):
        signature_column = candidate
        break

if signature_column is None:
    signature_overlap = set()

else:
    reference_signatures = set(
        reference_data[
            signature_column
        ]
        .dropna()
        .astype(str)
    )

    test_signatures = set(
        final_test_data[
            signature_column
        ]
        .dropna()
        .astype(str)
    )

    signature_overlap = (
        reference_signatures
        & test_signatures
    )

print(
    "Reference repositories:",
    sorted(
        reference_repositories
    ),
)

print(
    "Final-test repositories:",
    sorted(
        test_repositories
    ),
)

print(
    "Repository overlap:",
    len(
        repository_overlap
    ),
)

print(
    "Commit overlap:",
    len(
        commit_overlap
    ),
)

print(
    "Signature overlap:",
    len(
        signature_overlap
    ),
)

if repository_overlap:
    raise ValueError(
        "Repository leakage detected."
    )

if commit_overlap:
    raise ValueError(
        "Commit leakage detected."
    )

if signature_overlap:
    raise ValueError(
        "Change-signature leakage detected."
    )

del reference_data
del reference_frames

gc.collect()

print("Final-test disjointness checks: PASSED")

Reference repositories: ['kube_prometheus', 'kubernetes_examples', 'microservices_demo', 'terraform_aws_vpc']
Final-test repositories: ['ansible_examples', 'terraform_eks_blueprints']
Repository overlap: 0
Commit overlap: 0
Signature overlap: 0
Final-test disjointness checks: PASSED


In [16]:
def get_classifier_classes(
    model,
):
    if hasattr(
        model,
        "classes_",
    ):
        return np.asarray(
            model.classes_
        ).astype(str)

    if hasattr(
        model,
        "named_steps",
    ):
        for step_name in reversed(
            list(
                model.named_steps.keys()
            )
        ):
            step = model.named_steps[
                step_name
            ]

            if hasattr(
                step,
                "classes_",
            ):
                return np.asarray(
                    step.classes_
                ).astype(str)

    raise AttributeError(
        "The classifier classes could not "
        "be identified."
    )


def stable_softmax(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.ndim == 1:
        values = np.column_stack(
            [
                -values,
                values,
            ]
        )

    shifted = (
        values
        - np.max(
            values,
            axis=1,
            keepdims=True,
        )
    )

    exponentials = np.exp(
        shifted
    )

    return (
        exponentials
        / exponentials.sum(
            axis=1,
            keepdims=True,
        )
    )


def align_matrix_to_class_order(
    matrix,
    source_classes,
    target_classes,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    source_classes = [
        str(value).strip().lower()
        for value in source_classes
    ]

    target_classes = [
        str(value).strip().lower()
        for value in target_classes
    ]

    aligned_matrix = np.zeros(
        (
            matrix.shape[0],
            len(target_classes),
        ),
        dtype=np.float64,
    )

    for source_index, source_class in enumerate(
        source_classes
    ):
        if source_class not in target_classes:
            continue

        target_index = target_classes.index(
            source_class
        )

        aligned_matrix[
            :,
            target_index,
        ] = matrix[
            :,
            source_index,
        ]

    return aligned_matrix


def normalize_probability_matrix(
    matrix,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    matrix = np.clip(
        matrix,
        0.0,
        None,
    )

    zero_sum_rows = (
        matrix.sum(
            axis=1
        )
        <= 0
    )

    if zero_sum_rows.any():
        matrix[
            zero_sum_rows
        ] = (
            1.0
            / matrix.shape[1]
        )

    return (
        matrix
        / matrix.sum(
            axis=1,
            keepdims=True,
        )
    )


print("Model-output helpers loaded.")

Model-output helpers loaded.


In [17]:
def run_text_baseline_inference(
    dataframe,
):
    start_time = time.perf_counter()

    text_documents = pd.Series(
        baseline_text_builder(
            dataframe.copy()
        )
    ).fillna("").astype(str)

    if len(text_documents) != len(
        dataframe
    ):
        raise ValueError(
            "Text-builder output count does not "
            "match the input row count."
        )

    predicted_labels = (
        text_baseline_model.predict(
            text_documents
        )
    )

    model_classes = get_classifier_classes(
        text_baseline_model
    )

    if hasattr(
        text_baseline_model,
        "decision_function",
    ):
        raw_scores = (
            text_baseline_model
            .decision_function(
                text_documents
            )
        )

        aligned_scores = (
            align_matrix_to_class_order(
                raw_scores,
                model_classes,
                CLASS_ORDER,
            )
        )

        probabilities = stable_softmax(
            aligned_scores
        )

        probability_kind = (
            "decision_softmax_not_calibrated"
        )

    elif hasattr(
        text_baseline_model,
        "predict_proba",
    ):
        raw_probabilities = (
            text_baseline_model
            .predict_proba(
                text_documents
            )
        )

        probabilities = (
            align_matrix_to_class_order(
                raw_probabilities,
                model_classes,
                CLASS_ORDER,
            )
        )

        probabilities = (
            normalize_probability_matrix(
                probabilities
            )
        )

        aligned_scores = (
            probabilities.copy()
        )

        probability_kind = "probability"

    else:
        raise AttributeError(
            "The text model exposes neither "
            "decision_function nor predict_proba."
        )

    return {
        "predicted_labels":
            np.asarray(
                predicted_labels
            ).astype(str),

        "score_matrix":
            np.asarray(
                aligned_scores,
                dtype=np.float64,
            ),

        "probability_matrix":
            normalize_probability_matrix(
                probabilities
            ),

        "probability_kind":
            probability_kind,

        "inference_seconds":
            time.perf_counter()
            - start_time,
    }


print("Text inference function loaded.")

Text inference function loaded.


In [18]:
def run_structured_inference(
    dataframe,
):
    start_time = time.perf_counter()

    engineered_features = (
        engineer_structured_features(
            dataframe.copy()
        )
    )

    if len(engineered_features) != len(
        dataframe
    ):
        raise ValueError(
            "Structured feature count does not "
            "match the input row count."
        )

    predicted_labels = (
        structured_model.predict(
            engineered_features
        )
    )

    model_classes = get_classifier_classes(
        structured_model
    )

    if hasattr(
        structured_model,
        "predict_proba",
    ):
        raw_probabilities = (
            structured_model.predict_proba(
                engineered_features
            )
        )

        probabilities = (
            align_matrix_to_class_order(
                raw_probabilities,
                model_classes,
                CLASS_ORDER,
            )
        )

        probabilities = (
            normalize_probability_matrix(
                probabilities
            )
        )

        score_matrix = probabilities.copy()
        probability_kind = "probability"

    elif hasattr(
        structured_model,
        "decision_function",
    ):
        raw_scores = (
            structured_model
            .decision_function(
                engineered_features
            )
        )

        score_matrix = (
            align_matrix_to_class_order(
                raw_scores,
                model_classes,
                CLASS_ORDER,
            )
        )

        probabilities = stable_softmax(
            score_matrix
        )

        probability_kind = (
            "decision_softmax_not_calibrated"
        )

    else:
        raise AttributeError(
            "Structured model exposes neither "
            "predict_proba nor decision_function."
        )

    return {
        "predicted_labels":
            np.asarray(
                predicted_labels
            ).astype(str),

        "score_matrix":
            np.asarray(
                score_matrix,
                dtype=np.float64,
            ),

        "probability_matrix":
            normalize_probability_matrix(
                probabilities
            ),

        "probability_kind":
            probability_kind,

        "inference_seconds":
            time.perf_counter()
            - start_time,
    }


print("Structured inference function loaded.")

Structured inference function loaded.


In [20]:
def get_transformer_classes(
    model,
):
    id_to_label = getattr(
        model.config,
        "id2label",
        {},
    )

    candidate_classes = []

    for class_index in range(
        int(
            model.config.num_labels
        )
    ):
        candidate_label = id_to_label.get(
            class_index,
            id_to_label.get(
                str(
                    class_index
                ),
                class_index,
            ),
        )

        candidate_classes.append(
            str(
                candidate_label
            ).strip().lower()
        )

    if set(
        candidate_classes
    ) == set(
        CLASS_ORDER
    ):
        return candidate_classes

    return CLASS_ORDER.copy()


def run_transformer_inference(
    dataframe,
    batch_size=8,
):
    start_time = time.perf_counter()

    transformer_documents = pd.Series(
        transformer_text_builder(
            dataframe.copy()
        )
    ).fillna("").astype(str).tolist()

    if len(transformer_documents) != len(
        dataframe
    ):
        raise ValueError(
            "Transformer text count does not "
            "match the input row count."
        )

    probability_batches = []

    transformer_classes = (
        get_transformer_classes(
            transformer_model
        )
    )

    for batch_start in range(
        0,
        len(transformer_documents),
        batch_size,
    ):
        batch_documents = (
            transformer_documents[
                batch_start:
                batch_start
                + batch_size
            ]
        )

        encoded_batch = (
            transformer_tokenizer(
                batch_documents,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            )
        )

        encoded_batch = {
            key: value.to(
                DEVICE
            )
            for key, value
            in encoded_batch.items()
        }

        with torch.no_grad():
            model_output = (
                transformer_model(
                    **encoded_batch
                )
            )

            probability_batch = (
                torch.softmax(
                    model_output.logits.float(),
                    dim=1,
                )
                .detach()
                .cpu()
                .numpy()
            )

        probability_batches.append(
            probability_batch
        )

    raw_probabilities = np.vstack(
        probability_batches
    )

    probabilities = (
        align_matrix_to_class_order(
            raw_probabilities,
            transformer_classes,
            CLASS_ORDER,
        )
    )

    probabilities = (
        normalize_probability_matrix(
            probabilities
        )
    )

    predicted_ids = np.argmax(
        probabilities,
        axis=1,
    )

    predicted_labels = np.array(
        [
            CLASS_ORDER[
                class_index
            ]
            for class_index
            in predicted_ids
        ]
    )

    return {
        "predicted_labels":
            predicted_labels,

        "score_matrix":
            probabilities.copy(),

        "probability_matrix":
            probabilities,

        "probability_kind":
            "probability",

        "inference_seconds":
            time.perf_counter()
            - start_time,
    }


print("Transformer inference function loaded.")

Transformer inference function loaded.


In [21]:
print("=" * 72)
print("RUNNING FROZEN MODELS ON FINAL TEST")
print("=" * 72)

final_model_outputs = {}

print("\n1. Text baseline")

final_model_outputs[
    "text_baseline"
] = run_text_baseline_inference(
    final_test_data
)

print(
    "Completed in:",
    f"{final_model_outputs['text_baseline']['inference_seconds']:.2f}",
    "seconds",
)

print("\n2. Structured model")

final_model_outputs[
    "structured"
] = run_structured_inference(
    final_test_data
)

print(
    "Completed in:",
    f"{final_model_outputs['structured']['inference_seconds']:.2f}",
    "seconds",
)

print("\n3. Transformer")

final_model_outputs[
    "transformer"
] = run_transformer_inference(
    final_test_data,
    batch_size=8,
)

print(
    "Completed in:",
    f"{final_model_outputs['transformer']['inference_seconds']:.2f}",
    "seconds",
)

print("\nPrediction-length verification:")

for model_name, model_output in (
    final_model_outputs.items()
):
    prediction_count = len(
        model_output[
            "predicted_labels"
        ]
    )

    print(
        f"{model_name:<15}: "
        f"{prediction_count:,}"
    )

    if prediction_count != len(
        final_test_data
    ):
        raise ValueError(
            f"{model_name} produced an invalid "
            "prediction count."
        )

print("All final-test prediction counts: PASSED")

RUNNING FROZEN MODELS ON FINAL TEST

1. Text baseline
Completed in: 194.98 seconds

2. Structured model
Completed in: 10.78 seconds

3. Transformer
Completed in: 340.28 seconds

Prediction-length verification:
text_baseline  : 68,749
structured     : 68,749
transformer    : 68,749
All final-test prediction counts: PASSED


In [22]:
def calculate_binary_metrics(
    true_labels,
    predicted_labels,
    positive_labels,
):
    true_binary = np.isin(
        true_labels,
        positive_labels,
    ).astype(int)

    predicted_binary = np.isin(
        predicted_labels,
        positive_labels,
    ).astype(int)

    return {
        "precision": float(
            precision_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            )
        ),

        "recall": float(
            recall_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            )
        ),

        "f1": float(
            f1_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            )
        ),
    }


def calculate_macro_pr_auc(
    true_labels,
    score_matrix,
):
    average_precisions = []

    for class_index, class_name in enumerate(
        CLASS_ORDER
    ):
        binary_true = (
            np.asarray(
                true_labels
            )
            == class_name
        ).astype(int)

        if (
            binary_true.sum() == 0
            or binary_true.sum()
            == len(binary_true)
        ):
            continue

        average_precisions.append(
            average_precision_score(
                binary_true,
                score_matrix[
                    :,
                    class_index,
                ],
            )
        )

    if not average_precisions:
        return np.nan

    return float(
        np.mean(
            average_precisions
        )
    )


def calculate_model_metrics(
    model_name,
    predicted_labels,
    probability_matrix,
    inference_seconds=np.nan,
):
    labeled_mask_array = (
        final_test_labeled_mask
        .to_numpy()
    )

    true_labels = (
        final_test_data.loc[
            final_test_labeled_mask,
            "evaluation_label",
        ]
        .to_numpy(
            dtype=str
        )
    )

    labeled_predictions = np.asarray(
        predicted_labels
    )[
        labeled_mask_array
    ]

    labeled_probabilities = np.asarray(
        probability_matrix,
        dtype=np.float64,
    )[
        labeled_mask_array
    ]

    critical_metrics = (
        calculate_binary_metrics(
            true_labels,
            labeled_predictions,
            positive_labels=[
                "critical",
            ],
        )
    )

    high_critical_metrics = (
        calculate_binary_metrics(
            true_labels,
            labeled_predictions,
            positive_labels=[
                "high",
                "critical",
            ],
        )
    )

    return {
        "model": model_name,

        "records":
            int(
                len(
                    true_labels
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    true_labels,
                    labeled_predictions,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    true_labels,
                    labeled_predictions,
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    true_labels,
                    labeled_predictions,
                    labels=CLASS_ORDER,
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    true_labels,
                    labeled_predictions,
                    labels=CLASS_ORDER,
                    average="weighted",
                    zero_division=0,
                )
            ),

        "macro_pr_auc":
            calculate_macro_pr_auc(
                true_labels,
                labeled_probabilities,
            ),

        "log_loss":
            float(
                log_loss(
                    true_labels,
                    labeled_probabilities,
                    labels=CLASS_ORDER,
                )
            ),

        "critical_precision":
            critical_metrics[
                "precision"
            ],

        "critical_recall":
            critical_metrics[
                "recall"
            ],

        "critical_f1":
            critical_metrics[
                "f1"
            ],

        "high_critical_precision":
            high_critical_metrics[
                "precision"
            ],

        "high_critical_recall":
            high_critical_metrics[
                "recall"
            ],

        "high_critical_f1":
            high_critical_metrics[
                "f1"
            ],

        "inference_seconds":
            float(
                inference_seconds
            ),
    }


print("Final-evaluation helpers loaded.")

Final-evaluation helpers loaded.


In [23]:
final_predictions = (
    final_test_data.copy()
    .reset_index(drop=True)
)

raw_model_metric_records = []

for model_name, model_output in (
    final_model_outputs.items()
):
    final_predictions[
        f"{model_name}_prediction"
    ] = model_output[
        "predicted_labels"
    ]

    probability_matrix = (
        normalize_probability_matrix(
            model_output[
                "probability_matrix"
            ]
        )
    )

    final_predictions[
        f"{model_name}_confidence"
    ] = probability_matrix.max(
        axis=1
    )

    for class_index, class_name in enumerate(
        CLASS_ORDER
    ):
        final_predictions[
            f"{model_name}_score_{class_name}"
        ] = probability_matrix[
            :,
            class_index,
        ]

    raw_model_metric_records.append(
        calculate_model_metrics(
            model_name=model_name,

            predicted_labels=(
                model_output[
                    "predicted_labels"
                ]
            ),

            probability_matrix=(
                probability_matrix
            ),

            inference_seconds=(
                model_output[
                    "inference_seconds"
                ]
            ),
        )
    )

raw_model_results = pd.DataFrame(
    raw_model_metric_records
)

raw_model_results = (
    raw_model_results
    .sort_values(
        [
            "macro_f1",
            "critical_recall",
            "high_critical_recall",
            "macro_pr_auc",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

raw_model_results.insert(
    0,
    "reporting_rank",
    np.arange(
        1,
        len(
            raw_model_results
        )
        + 1,
    ),
)

display(
    raw_model_results
)

print(
    "\nThe final-test ranking is for reporting only."
)

print(
    "It does not change the frozen primary model:",
    PRIMARY_MODEL_NAME,
)

C:\Users\Lenovo\anaconda3\envs\driftguard\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['benign', 'low', 'medium', 'high', 'critical']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['benign', 'critical', 'high', 'low', 'medium'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(
C:\Users\Lenovo\anaconda3\envs\driftguard\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['benign', 'low', 'medium', 'high', 'critical']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['benign', 'critical', 'high', 'low', 'medium'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(
C:\Users\Lenovo\anaconda3\envs\driftguard\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['benign', 'low', 'medium', 'high', 'critical']. But this function

,reporting_rank,model,records,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_pr_auc,log_loss,critical_precision,critical_recall,critical_f1,high_critical_precision,high_critical_recall,high_critical_f1,inference_seconds
0,1,text_baseline,61899,0.591415,0.516079,0.389371,0.623980,0.449601,1.331315,0.028450,0.062500,0.039101,0.056801,0.228,0.090945,194.984574
1,2,structured,61899,0.515727,0.419490,0.338477,0.548732,0.356153,2.940574,0.053145,0.153125,0.078905,0.075088,0.170,0.104167,10.777380
2,3,transformer,61899,0.437568,0.459366,0.310426,0.461793,0.406193,6.271581,0.045025,0.253125,0.076451,0.059574,0.364,0.102391,340.275178



The final-test ranking is for reporting only.
It does not change the frozen primary model: structured


In [24]:
print("=" * 72)
print("APPLYING FROZEN DETERMINISTIC RULES")
print("=" * 72)

final_rule_results = (
    final_predictions
    .apply(
        apply_deterministic_rules_to_row,
        axis=1,
        result_type="expand",
    )
)

final_rule_results = (
    final_rule_results.rename(
        columns={
            "matched_rule_ids":
                "deterministic_rule_ids",
        }
    )
)

final_hybrid_data = pd.concat(
    [
        final_predictions.reset_index(
            drop=True
        ),

        final_rule_results.reset_index(
            drop=True
        ),
    ],
    axis=1,
)

if final_hybrid_data.columns.duplicated().any():
    duplicate_names = (
        final_hybrid_data.columns[
            final_hybrid_data.columns.duplicated()
        ].tolist()
    )

    raise ValueError(
        "Duplicate columns after final rule "
        f"processing: {duplicate_names}"
    )

print(
    "Final-test rule matches:",
    f"{final_hybrid_data['rule_match_count'].gt(0).sum():,}",
)

print(
    "Critical rule records:",
    f"{final_hybrid_data['rule_severity'].eq('critical').sum():,}",
)

print(
    "High rule records:",
    f"{final_hybrid_data['rule_severity'].eq('high').sum():,}",
)

print(
    "Medium rule records:",
    f"{final_hybrid_data['rule_severity'].eq('medium').sum():,}",
)

APPLYING FROZEN DETERMINISTIC RULES
Final-test rule matches: 3,072
Critical rule records: 2,440
High rule records: 67
Medium rule records: 565


In [25]:
MODEL_NAMES = [
    "structured",
    "transformer",
    "text_baseline",
]


def extract_model_probability_matrix(
    dataframe,
    model_name,
):
    probability_columns = [
        f"{model_name}_score_{class_name}"
        for class_name in CLASS_ORDER
    ]

    probability_matrix = (
        dataframe[
            probability_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    return normalize_probability_matrix(
        probability_matrix
    )


FINAL_MODEL_PROBABILITY_MATRICES = {
    model_name:
        extract_model_probability_matrix(
            final_hybrid_data,
            model_name,
        )
    for model_name in MODEL_NAMES
}

weighted_ensemble_probabilities = np.zeros(
    (
        len(
            final_hybrid_data
        ),
        len(
            CLASS_ORDER
        ),
    ),
    dtype=np.float64,
)

for model_name, model_weight in (
    MODEL_WEIGHTS.items()
):
    weighted_ensemble_probabilities += (
        float(
            model_weight
        )
        * FINAL_MODEL_PROBABILITY_MATRICES[
            model_name
        ]
    )

weighted_ensemble_probabilities = (
    normalize_probability_matrix(
        weighted_ensemble_probabilities
    )
)

weighted_ensemble_ids = np.argmax(
    weighted_ensemble_probabilities,
    axis=1,
)

final_hybrid_data[
    "weighted_ensemble_prediction"
] = [
    CLASS_ORDER[
        class_index
    ]
    for class_index
    in weighted_ensemble_ids
]

final_hybrid_data[
    "weighted_ensemble_confidence"
] = weighted_ensemble_probabilities.max(
    axis=1
)

for class_index, class_name in enumerate(
    CLASS_ORDER
):
    final_hybrid_data[
        f"weighted_ensemble_score_{class_name}"
    ] = weighted_ensemble_probabilities[
        :,
        class_index,
    ]

print(
    final_hybrid_data[
        "weighted_ensemble_prediction"
    ]
    .value_counts()
    .reindex(
        CLASS_ORDER,
        fill_value=0,
    )
)

weighted_ensemble_prediction
benign      24713
low         41455
medium         85
high         1112
critical     1384
Name: count, dtype: int64


In [26]:
model_prediction_columns = [
    f"{model_name}_prediction"
    for model_name in MODEL_NAMES
]

model_prediction_frame = (
    final_hybrid_data[
        model_prediction_columns
    ]
    .fillna("benign")
    .astype(str)
    .apply(
        lambda column:
        column.str.strip().str.lower()
    )
)

model_prediction_ranks = (
    model_prediction_frame.replace(
        RISK_RANK
    )
)

final_hybrid_data[
    "model_unique_prediction_count"
] = model_prediction_frame.nunique(
    axis=1
)

final_hybrid_data[
    "model_three_way_disagreement"
] = (
    final_hybrid_data[
        "model_unique_prediction_count"
    ]
    == 3
)

final_hybrid_data[
    "model_minimum_risk_rank"
] = model_prediction_ranks.min(
    axis=1
)

final_hybrid_data[
    "model_maximum_risk_rank"
] = model_prediction_ranks.max(
    axis=1
)

final_hybrid_data[
    "model_risk_spread"
] = (
    final_hybrid_data[
        "model_maximum_risk_rank"
    ]
    - final_hybrid_data[
        "model_minimum_risk_rank"
    ]
)

high_critical_vote_matrix = (
    model_prediction_frame.isin(
        [
            "high",
            "critical",
        ]
    )
)

critical_vote_matrix = (
    model_prediction_frame.eq(
        "critical"
    )
)

final_hybrid_data[
    "high_critical_model_votes"
] = high_critical_vote_matrix.sum(
    axis=1
)

final_hybrid_data[
    "critical_model_votes"
] = critical_vote_matrix.sum(
    axis=1
)

final_hybrid_data[
    "ensemble_high_critical_probability"
] = (
    final_hybrid_data[
        "weighted_ensemble_score_high"
    ]
    + final_hybrid_data[
        "weighted_ensemble_score_critical"
    ]
)

print(
    "Three-way disagreements:",
    f"{final_hybrid_data['model_three_way_disagreement'].sum():,}",
)

print(
    "Two-or-more high/critical votes:",
    f"{final_hybrid_data['high_critical_model_votes'].ge(2).sum():,}",
)

Three-way disagreements: 1,974
Two-or-more high/critical votes: 1,259


In [27]:
final_hybrid_decisions = (
    final_hybrid_data
    .apply(
        build_hybrid_decision,
        axis=1,
        result_type="expand",
    )
)

final_hybrid_data = pd.concat(
    [
        final_hybrid_data,
        final_hybrid_decisions,
    ],
    axis=1,
)

rules_plus_ensemble_predictions = []

for _, row in final_hybrid_data.iterrows():
    ensemble_rank = RISK_RANK[
        row[
            "weighted_ensemble_prediction"
        ]
    ]

    rule_rank = int(
        row[
            "rule_severity_rank"
        ]
    )

    rules_plus_ensemble_predictions.append(
        RANK_TO_CLASS[
            max(
                ensemble_rank,
                rule_rank,
            )
        ]
    )

final_hybrid_data[
    "rules_plus_ensemble_prediction"
] = rules_plus_ensemble_predictions

rules_plus_ensemble_probabilities = (
    adjust_scores_to_decisions(
        weighted_ensemble_probabilities,

        final_hybrid_data[
            "rules_plus_ensemble_prediction"
        ].tolist(),

        boost=(
            HYBRID_SETTINGS[
                "hybrid_score_boost"
            ]
        ),
    )
)

safety_hybrid_probabilities = (
    adjust_scores_to_decisions(
        weighted_ensemble_probabilities,

        final_hybrid_data[
            "safety_hybrid_prediction"
        ].tolist(),

        boost=(
            HYBRID_SETTINGS[
                "hybrid_score_boost"
            ]
        ),
    )
)

rules_plus_ensemble_probabilities = (
    normalize_probability_matrix(
        rules_plus_ensemble_probabilities
    )
)

safety_hybrid_probabilities = (
    normalize_probability_matrix(
        safety_hybrid_probabilities
    )
)

final_hybrid_data[
    "safety_hybrid_confidence"
] = safety_hybrid_probabilities.max(
    axis=1
)

for class_index, class_name in enumerate(
    CLASS_ORDER
):
    final_hybrid_data[
        f"safety_hybrid_score_{class_name}"
    ] = safety_hybrid_probabilities[
        :,
        class_index,
    ]

print(
    "Rule overrides:",
    f"{final_hybrid_data['rule_override_applied'].sum():,}",
)

print(
    "Consensus escalations:",
    f"{final_hybrid_data['model_consensus_escalation'].sum():,}",
)

Rule overrides: 2,825
Consensus escalations: 314


In [28]:
final_labeled_mask_array = (
    final_test_labeled_mask
    .to_numpy()
)

final_true_labels = (
    final_hybrid_data.loc[
        final_test_labeled_mask,
        "evaluation_label",
    ]
    .to_numpy(
        dtype=str
    )
)


def calculate_variant_metrics(
    variant_name,
    predicted_labels,
    probability_matrix,
):
    labeled_predictions = np.asarray(
        predicted_labels
    )[
        final_labeled_mask_array
    ]

    labeled_probabilities = np.asarray(
        probability_matrix,
        dtype=np.float64,
    )[
        final_labeled_mask_array
    ]

    critical_metrics = (
        calculate_binary_metrics(
            final_true_labels,
            labeled_predictions,
            positive_labels=[
                "critical",
            ],
        )
    )

    high_critical_metrics = (
        calculate_binary_metrics(
            final_true_labels,
            labeled_predictions,
            positive_labels=[
                "high",
                "critical",
            ],
        )
    )

    return {
        "variant":
            variant_name,

        "records":
            int(
                len(
                    final_true_labels
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    final_true_labels,
                    labeled_predictions,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    final_true_labels,
                    labeled_predictions,
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    final_true_labels,
                    labeled_predictions,
                    labels=CLASS_ORDER,
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    final_true_labels,
                    labeled_predictions,
                    labels=CLASS_ORDER,
                    average="weighted",
                    zero_division=0,
                )
            ),

        "macro_pr_auc":
            calculate_macro_pr_auc(
                final_true_labels,
                labeled_probabilities,
            ),

        "log_loss":
            float(
                log_loss(
                    final_true_labels,
                    labeled_probabilities,
                    labels=CLASS_ORDER,
                )
            ),

        "critical_precision":
            critical_metrics[
                "precision"
            ],

        "critical_recall":
            critical_metrics[
                "recall"
            ],

        "critical_f1":
            critical_metrics[
                "f1"
            ],

        "high_critical_precision":
            high_critical_metrics[
                "precision"
            ],

        "high_critical_recall":
            high_critical_metrics[
                "recall"
            ],

        "high_critical_f1":
            high_critical_metrics[
                "f1"
            ],
    }


final_variant_results = pd.DataFrame(
    [
        calculate_variant_metrics(
            variant_name=(
                f"primary_{PRIMARY_MODEL_NAME}"
            ),

            predicted_labels=(
                final_hybrid_data[
                    f"{PRIMARY_MODEL_NAME}_prediction"
                ].to_numpy()
            ),

            probability_matrix=(
                FINAL_MODEL_PROBABILITY_MATRICES[
                    PRIMARY_MODEL_NAME
                ]
            ),
        ),

        calculate_variant_metrics(
            variant_name="weighted_ensemble",

            predicted_labels=(
                final_hybrid_data[
                    "weighted_ensemble_prediction"
                ].to_numpy()
            ),

            probability_matrix=(
                weighted_ensemble_probabilities
            ),
        ),

        calculate_variant_metrics(
            variant_name="rules_plus_ensemble",

            predicted_labels=(
                final_hybrid_data[
                    "rules_plus_ensemble_prediction"
                ].to_numpy()
            ),

            probability_matrix=(
                rules_plus_ensemble_probabilities
            ),
        ),

        calculate_variant_metrics(
            variant_name="safety_hybrid",

            predicted_labels=(
                final_hybrid_data[
                    "safety_hybrid_prediction"
                ].to_numpy()
            ),

            probability_matrix=(
                safety_hybrid_probabilities
            ),
        ),
    ]
)

final_variant_results[
    "production_variant"
] = final_variant_results[
    "variant"
].eq(
    PRODUCTION_VARIANT
)

display(
    final_variant_results
)

production_final_row = (
    final_variant_results[
        final_variant_results[
            "variant"
        ].eq(
            PRODUCTION_VARIANT
        )
    ]
    .iloc[0]
)

C:\Users\Lenovo\anaconda3\envs\driftguard\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['benign', 'low', 'medium', 'high', 'critical']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['benign', 'critical', 'high', 'low', 'medium'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(
C:\Users\Lenovo\anaconda3\envs\driftguard\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['benign', 'low', 'medium', 'high', 'critical']. But this function assumes labels are ordered lexicographically. Pass the ordered labels=['benign', 'critical', 'high', 'low', 'medium'] and ensure that the columns of y_prob correspond to this ordering.
  warnings.warn(
C:\Users\Lenovo\anaconda3\envs\driftguard\Lib\site-packages\sklearn\metrics\_classification.py:192: UserWarning: Labels passed were ['benign', 'low', 'medium', 'high', 'critical']. But this function

,variant,records,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_pr_auc,log_loss,critical_precision,critical_recall,critical_f1,high_critical_precision,high_critical_recall,high_critical_f1,production_variant
0,primary_structured,61899,0.515727,0.419490,0.338477,0.548732,0.356179,2.940574,0.053145,0.153125,0.078905,0.075088,0.170,0.104167,False
1,weighted_ensemble,61899,0.445645,0.473511,0.343073,0.467687,0.461875,1.569128,0.044798,0.193750,0.072770,0.067936,0.314,0.111704,False
2,rules_plus_ensemble,61899,0.424062,0.619747,0.273441,0.456302,0.464210,1.761253,0.051136,0.590625,0.094124,0.063747,0.588,0.115023,False
3,safety_hybrid,61899,0.423028,0.623216,0.269263,0.455773,0.463909,1.760715,0.050821,0.600000,0.093704,0.061425,0.600,0.111441,True


In [29]:
severity_anchor_vector = np.array(
    [
        SEVERITY_ANCHORS[
            class_name
        ]
        for class_name in CLASS_ORDER
    ],
    dtype=np.float64,
)

expected_probability_risk = (
    safety_hybrid_probabilities
    @ severity_anchor_vector
)

decision_anchor = (
    final_hybrid_data[
        "safety_hybrid_prediction"
    ]
    .map(
        SEVERITY_ANCHORS
    )
    .fillna(
        SEVERITY_ANCHORS[
            "benign"
        ]
    )
    .to_numpy(
        dtype=np.float64
    )
)

score_weights = SCORING_SETTINGS[
    "score_weights"
]

model_risk_component = (
    float(
        score_weights[
            "decision_anchor"
        ]
    )
    * decision_anchor

    + float(
        score_weights[
            "expected_probability_risk"
        ]
    )
    * expected_probability_risk
)

rule_anchor = (
    final_hybrid_data[
        "rule_severity"
    ]
    .map(
        SEVERITY_ANCHORS
    )
    .fillna(0.0)
    .to_numpy(
        dtype=np.float64
    )
)

rule_confidence = (
    pd.to_numeric(
        final_hybrid_data[
            "rule_confidence"
        ],
        errors="coerce",
    )
    .fillna(0.0)
    .clip(
        lower=0.0,
        upper=1.0,
    )
    .to_numpy(
        dtype=np.float64
    )
)

rule_match_mask = (
    pd.to_numeric(
        final_hybrid_data[
            "rule_match_count"
        ],
        errors="coerce",
    )
    .fillna(0)
    .gt(0)
    .to_numpy()
)

rule_risk_floor = np.where(
    rule_match_mask,

    rule_anchor
    * (
        0.65
        + 0.35
        * rule_confidence
    ),

    0.0,
)

critical_votes = (
    final_hybrid_data[
        "critical_model_votes"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

high_critical_votes = (
    final_hybrid_data[
        "high_critical_model_votes"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

high_critical_probability = (
    final_hybrid_data[
        "ensemble_high_critical_probability"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

consensus_bonus = np.select(
    [
        critical_votes >= 2,

        high_critical_votes >= 2,

        (
            critical_votes >= 1
        )
        & (
            high_critical_probability
            >= SCORING_SETTINGS[
                "supported_critical_probability"
            ]
        ),
    ],
    [
        SCORING_SETTINGS[
            "consensus_bonus"
        ][
            "two_critical_votes"
        ],

        SCORING_SETTINGS[
            "consensus_bonus"
        ][
            "two_high_critical_votes"
        ],

        SCORING_SETTINGS[
            "consensus_bonus"
        ][
            "supported_single_critical_vote"
        ],
    ],
    default=0.0,
)

predicted_risk_rank = (
    final_hybrid_data[
        "safety_hybrid_prediction"
    ]
    .map(
        RISK_RANK
    )
    .fillna(0)
    .to_numpy(
        dtype=np.int64
    )
)

removal_mask = (
    final_hybrid_data[
        "operation"
    ]
    .apply(
        operation_is_removal
    )
    .to_numpy()
)

addition_mask = (
    final_hybrid_data[
        "operation"
    ]
    .apply(
        operation_is_addition
    )
    .to_numpy()
)

operation_bonus = np.zeros(
    len(
        final_hybrid_data
    ),
    dtype=np.float64,
)

operation_bonus[
    removal_mask
    & (
        predicted_risk_rank >= 2
    )
] += (
    SCORING_SETTINGS[
        "operation_bonus"
    ][
        "security_relevant_removal"
    ]
)

operation_bonus[
    addition_mask
    & (
        predicted_risk_rank >= 3
    )
] += (
    SCORING_SETTINGS[
        "operation_bonus"
    ][
        "high_risk_addition"
    ]
)

change_risk_score = np.maximum(
    model_risk_component,
    rule_risk_floor,
)

change_risk_score = np.clip(
    change_risk_score
    + consensus_bonus
    + operation_bonus,
    0.0,
    100.0,
)

final_hybrid_data[
    "expected_probability_risk"
] = expected_probability_risk

final_hybrid_data[
    "decision_anchor_score"
] = decision_anchor

final_hybrid_data[
    "model_risk_component"
] = model_risk_component

final_hybrid_data[
    "rule_risk_floor"
] = rule_risk_floor

final_hybrid_data[
    "consensus_score_bonus"
] = consensus_bonus

final_hybrid_data[
    "operation_score_bonus"
] = operation_bonus

final_hybrid_data[
    "change_risk_score"
] = change_risk_score

print(
    "Mean change score:",
    f"{change_risk_score.mean():.4f}",
)

print(
    "Maximum change score:",
    f"{change_risk_score.max():.4f}",
)

Mean change score: 24.8465
Maximum change score: 100.0000


In [30]:
entropy_values = normalized_entropy(
    safety_hybrid_probabilities
)

uncertainty_score = (
    entropy_values
    * 100.0
)

model_unique_count = (
    final_hybrid_data[
        "model_unique_prediction_count"
    ]
    .clip(
        lower=1,
        upper=3,
    )
    .to_numpy(
        dtype=np.float64
    )
)

agreement_score = (
    1.0
    - (
        model_unique_count
        - 1.0
    )
    / 2.0
)

hybrid_confidence = (
    final_hybrid_data[
        "safety_hybrid_confidence"
    ]
    .clip(
        lower=0.0,
        upper=1.0,
    )
    .to_numpy(
        dtype=np.float64
    )
)

score_confidence = np.where(
    rule_match_mask,

    (
        0.50
        * hybrid_confidence
        + 0.25
        * agreement_score
        + 0.25
        * rule_confidence
    ),

    (
        0.70
        * hybrid_confidence
        + 0.30
        * agreement_score
    ),
)

score_confidence = np.clip(
    score_confidence
    * 100.0,
    0.0,
    100.0,
)

risk_spread = (
    final_hybrid_data[
        "model_risk_spread"
    ]
    .clip(
        lower=0,
        upper=4,
    )
    .to_numpy(
        dtype=np.float64
    )
)

rule_override = (
    final_hybrid_data[
        "rule_override_applied"
    ]
    .fillna(False)
    .astype(bool)
    .to_numpy()
)

operational_review_priority = (
    0.75
    * change_risk_score

    + 0.25
    * uncertainty_score

    + 5.0
    * risk_spread

    + np.where(
        rule_override,
        5.0,
        0.0,
    )
)

operational_review_priority = np.clip(
    operational_review_priority,
    0.0,
    100.0,
)

final_hybrid_data[
    "probability_entropy"
] = entropy_values

final_hybrid_data[
    "uncertainty_score"
] = uncertainty_score

final_hybrid_data[
    "model_agreement_score"
] = (
    agreement_score
    * 100.0
)

final_hybrid_data[
    "score_confidence"
] = score_confidence

final_hybrid_data[
    "operational_review_priority"
] = operational_review_priority

final_hybrid_data[
    "drift_band"
] = final_hybrid_data[
    "change_risk_score"
].apply(
    assign_drift_band
)

final_hybrid_data[
    "score_predicted_label"
] = final_hybrid_data[
    "drift_band"
].map(
    BAND_TO_LABEL
)

final_hybrid_data[
    "score_explanation"
] = final_hybrid_data.apply(
    build_score_explanation,
    axis=1,
)

final_score_distribution = (
    final_hybrid_data[
        "drift_band"
    ]
    .value_counts()
    .reindex(
        [
            "stable",
            "watch",
            "concerning",
            "high",
            "critical",
        ],
        fill_value=0,
    )
    .rename_axis("drift_band")
    .reset_index(name="records")
)

final_score_distribution[
    "percentage"
] = (
    final_score_distribution[
        "records"
    ]
    / len(
        final_hybrid_data
    )
    * 100.0
)

display(
    final_score_distribution
)

,drift_band,records,percentage
0,stable,24064,35.002691
1,watch,39005,56.735371
2,concerning,596,0.866922
3,high,1262,1.835663
4,critical,3822,5.559354


In [31]:
score_true_labels = (
    final_hybrid_data.loc[
        final_test_labeled_mask,
        "evaluation_label",
    ]
    .to_numpy(
        dtype=str
    )
)

score_predicted_labels = (
    final_hybrid_data.loc[
        final_test_labeled_mask,
        "score_predicted_label",
    ]
    .to_numpy(
        dtype=str
    )
)

labeled_change_scores = (
    final_hybrid_data.loc[
        final_test_labeled_mask,
        "change_risk_score",
    ]
    .to_numpy(
        dtype=np.float64
    )
)

true_anchor_scores = np.array(
    [
        SEVERITY_ANCHORS[
            label
        ]
        for label in score_true_labels
    ],
    dtype=np.float64,
)

rank_correlation = float(
    pd.Series(
        true_anchor_scores
    )
    .rank(
        method="average"
    )
    .corr(
        pd.Series(
            labeled_change_scores
        )
        .rank(
            method="average"
        )
    )
)

critical_true = (
    score_true_labels
    == "critical"
).astype(int)

critical_prediction = (
    labeled_change_scores
    >= 80.0
).astype(int)

high_critical_true = np.isin(
    score_true_labels,
    [
        "high",
        "critical",
    ],
).astype(int)

high_critical_prediction = (
    labeled_change_scores
    >= 60.0
).astype(int)

final_score_metrics = {
    "labeled_records":
        int(
            len(
                score_true_labels
            )
        ),

    "accuracy":
        float(
            accuracy_score(
                score_true_labels,
                score_predicted_labels,
            )
        ),

    "balanced_accuracy":
        float(
            balanced_accuracy_score(
                score_true_labels,
                score_predicted_labels,
            )
        ),

    "macro_f1":
        float(
            f1_score(
                score_true_labels,
                score_predicted_labels,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0,
            )
        ),

    "weighted_f1":
        float(
            f1_score(
                score_true_labels,
                score_predicted_labels,
                labels=CLASS_ORDER,
                average="weighted",
                zero_division=0,
            )
        ),

    "severity_rank_correlation":
        rank_correlation,

    "mean_absolute_anchor_error":
        float(
            np.mean(
                np.abs(
                    labeled_change_scores
                    - true_anchor_scores
                )
            )
        ),

    "critical_precision":
        float(
            precision_score(
                critical_true,
                critical_prediction,
                zero_division=0,
            )
        ),

    "critical_recall":
        float(
            recall_score(
                critical_true,
                critical_prediction,
                zero_division=0,
            )
        ),

    "critical_f1":
        float(
            f1_score(
                critical_true,
                critical_prediction,
                zero_division=0,
            )
        ),

    "high_critical_precision":
        float(
            precision_score(
                high_critical_true,
                high_critical_prediction,
                zero_division=0,
            )
        ),

    "high_critical_recall":
        float(
            recall_score(
                high_critical_true,
                high_critical_prediction,
                zero_division=0,
            )
        ),

    "high_critical_f1":
        float(
            f1_score(
                high_critical_true,
                high_critical_prediction,
                zero_division=0,
            )
        ),
}

display(
    pd.DataFrame(
        [
            final_score_metrics
        ]
    )
)

,labeled_records,accuracy,balanced_accuracy,macro_f1,weighted_f1,severity_rank_correlation,mean_absolute_anchor_error,critical_precision,critical_recall,critical_f1,high_critical_precision,high_critical_recall,high_critical_f1
0,61899,0.420734,0.569791,0.253899,0.453516,0.273894,17.274685,0.050235,0.6,0.092709,0.061425,0.6,0.111441


In [32]:
production_predictions = (
    final_hybrid_data.loc[
        final_test_labeled_mask,
        "safety_hybrid_prediction",
    ]
    .to_numpy(
        dtype=str
    )
)

production_probabilities = (
    safety_hybrid_probabilities[
        final_labeled_mask_array
    ]
)

final_per_class_records = []

for class_index, class_name in enumerate(
    CLASS_ORDER
):
    binary_true = (
        final_true_labels
        == class_name
    ).astype(int)

    binary_prediction = (
        production_predictions
        == class_name
    ).astype(int)

    if (
        binary_true.sum() > 0
        and binary_true.sum()
        < len(binary_true)
    ):
        average_precision = float(
            average_precision_score(
                binary_true,
                production_probabilities[
                    :,
                    class_index,
                ],
            )
        )
    else:
        average_precision = np.nan

    final_per_class_records.append(
        {
            "class":
                class_name,

            "support":
                int(
                    binary_true.sum()
                ),

            "precision":
                float(
                    precision_score(
                        binary_true,
                        binary_prediction,
                        zero_division=0,
                    )
                ),

            "recall":
                float(
                    recall_score(
                        binary_true,
                        binary_prediction,
                        zero_division=0,
                    )
                ),

            "f1":
                float(
                    f1_score(
                        binary_true,
                        binary_prediction,
                        zero_division=0,
                    )
                ),

            "average_precision":
                average_precision,
        }
    )

final_per_class_metrics = (
    pd.DataFrame(
        final_per_class_records
    )
)

critical_false_negative_mask = (
    final_test_labeled_mask
    &
    final_hybrid_data[
        "evaluation_label"
    ].eq("critical")
    &
    ~final_hybrid_data[
        "safety_hybrid_prediction"
    ].eq("critical")
)

high_critical_false_negative_mask = (
    final_test_labeled_mask
    &
    final_hybrid_data[
        "evaluation_label"
    ].isin(
        [
            "high",
            "critical",
        ]
    )
    &
    ~final_hybrid_data[
        "safety_hybrid_prediction"
    ].isin(
        [
            "high",
            "critical",
        ]
    )
)

high_critical_false_positive_mask = (
    final_test_labeled_mask
    &
    ~final_hybrid_data[
        "evaluation_label"
    ].isin(
        [
            "high",
            "critical",
        ]
    )
    &
    final_hybrid_data[
        "safety_hybrid_prediction"
    ].isin(
        [
            "high",
            "critical",
        ]
    )
)

final_critical_false_negatives = (
    final_hybrid_data[
        critical_false_negative_mask
    ]
    .copy()
)

final_high_critical_false_negatives = (
    final_hybrid_data[
        high_critical_false_negative_mask
    ]
    .copy()
)

final_high_critical_false_positives = (
    final_hybrid_data[
        high_critical_false_positive_mask
    ]
    .copy()
)

display(
    final_per_class_metrics
)

print(
    "Critical false negatives:",
    f"{len(final_critical_false_negatives):,}",
)

print(
    "High/critical false negatives:",
    f"{len(final_high_critical_false_negatives):,}",
)

print(
    "High/critical false positives:",
    f"{len(final_high_critical_false_positives):,}",
)

,class,support,precision,recall,f1,average_precision
0,benign,46499,0.862432,0.334631,0.482174,0.863491
1,low,14816,0.267360,0.692562,0.385788,0.413601
2,medium,84,0.141414,1.000000,0.247788,0.724499
3,high,180,0.079566,0.488889,0.136858,0.253254
4,critical,320,0.050821,0.600000,0.093704,0.064697


Critical false negatives: 128
High/critical false negatives: 200
High/critical false positives: 4,584


In [33]:
final_hybrid_data[
    "commit_timestamp"
] = pd.to_datetime(
    final_hybrid_data[
        "commit_author_date"
    ],
    errors="coerce",
    utc=True,
)

if final_hybrid_data[
    "commit_timestamp"
].isna().any():
    final_hybrid_data[
        "commit_timestamp"
    ] = (
        final_hybrid_data
        .groupby(
            "repository"
        )[
            "commit_timestamp"
        ]
        .transform(
            lambda series:
            series.ffill().bfill()
        )
    )

final_commit_records = []

for (
    repository,
    commit_hash,
), group in final_hybrid_data.groupby(
    [
        "repository",
        "commit_hash",
    ],
    sort=False,
    dropna=False,
):
    aggregate_scores = (
        aggregate_event_scores(
            group[
                "change_risk_score"
            ].to_numpy()
        )
    )

    confidence_weights = np.maximum(
        group[
            "change_risk_score"
        ].to_numpy(
            dtype=np.float64
        ),
        1.0,
    )

    final_commit_records.append(
        {
            "repository":
                repository,

            "commit_hash":
                commit_hash,

            "commit_timestamp":
                group[
                    "commit_timestamp"
                ].min(),

            "commit_message":
                group[
                    "commit_message"
                ].iloc[0],

            "changed_fields":
                int(
                    len(
                        group
                    )
                ),

            "changed_files":
                int(
                    group[
                        "file_path"
                    ].nunique()
                ),

            "risky_change_count":
                int(
                    (
                        group[
                            "change_risk_score"
                        ]
                        >= 35.0
                    ).sum()
                ),

            "high_change_count":
                int(
                    (
                        group[
                            "change_risk_score"
                        ]
                        >= 60.0
                    ).sum()
                ),

            "critical_change_count":
                int(
                    (
                        group[
                            "change_risk_score"
                        ]
                        >= 80.0
                    ).sum()
                ),

            "rule_matched_change_count":
                int(
                    (
                        group[
                            "rule_match_count"
                        ]
                        > 0
                    ).sum()
                ),

            "maximum_change_score":
                aggregate_scores[
                    "maximum_score"
                ],

            "top_three_mean_score":
                aggregate_scores[
                    "top_three_mean_score"
                ],

            "risk_mass_score":
                aggregate_scores[
                    "risk_mass_score"
                ],

            "commit_drift_score":
                aggregate_scores[
                    "event_score"
                ],

            "commit_score_confidence":
                float(
                    np.average(
                        group[
                            "score_confidence"
                        ],
                        weights=confidence_weights,
                    )
                ),

            "maximum_review_priority":
                float(
                    group[
                        "operational_review_priority"
                    ].max()
                ),
        }
    )

final_commit_scores = pd.DataFrame(
    final_commit_records
)

final_commit_scores[
    "commit_drift_band"
] = final_commit_scores[
    "commit_drift_score"
].apply(
    assign_drift_band
)

final_commit_scores = (
    final_commit_scores
    .sort_values(
        [
            "repository",
            "commit_timestamp",
            "commit_hash",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Final-test commits scored:",
    f"{len(final_commit_scores):,}",
)

Final-test commits scored: 1,252


In [34]:
final_file_event_records = []

for (
    repository,
    file_path,
    commit_hash,
), group in final_hybrid_data.groupby(
    [
        "repository",
        "file_path",
        "commit_hash",
    ],
    sort=False,
    dropna=False,
):
    aggregate_scores = (
        aggregate_event_scores(
            group[
                "change_risk_score"
            ].to_numpy()
        )
    )

    final_file_event_records.append(
        {
            "repository":
                repository,

            "file_path":
                file_path,

            "commit_hash":
                commit_hash,

            "commit_timestamp":
                group[
                    "commit_timestamp"
                ].min(),

            "field_change_count":
                int(
                    len(
                        group
                    )
                ),

            "high_or_critical_changes":
                int(
                    (
                        group[
                            "change_risk_score"
                        ]
                        >= 60.0
                    ).sum()
                ),

            "rule_matched_changes":
                int(
                    (
                        group[
                            "rule_match_count"
                        ]
                        > 0
                    ).sum()
                ),

            "maximum_change_score":
                aggregate_scores[
                    "maximum_score"
                ],

            "file_event_drift_score":
                aggregate_scores[
                    "event_score"
                ],
        }
    )

final_file_event_scores = pd.DataFrame(
    final_file_event_records
)

final_file_event_scores = (
    final_file_event_scores
    .sort_values(
        [
            "repository",
            "file_path",
            "commit_timestamp",
            "commit_hash",
        ]
    )
    .reset_index(drop=True)
)

final_file_timeline = (
    calculate_decayed_pressure(
        dataframe=(
            final_file_event_scores
        ),

        group_columns=[
            "repository",
            "file_path",
        ],

        timestamp_column=(
            "commit_timestamp"
        ),

        event_score_column=(
            "file_event_drift_score"
        ),

        half_life_days=(
            SCORING_SETTINGS[
                "file_pressure_half_life_days"
            ]
        ),
    )
)

final_repository_timeline = (
    calculate_decayed_pressure(
        dataframe=(
            final_commit_scores
        ),

        group_columns=[
            "repository",
        ],

        timestamp_column=(
            "commit_timestamp"
        ),

        event_score_column=(
            "commit_drift_score"
        ),

        half_life_days=(
            SCORING_SETTINGS[
                "repository_pressure_half_life_days"
            ]
        ),
    )
)

final_file_snapshot = (
    final_file_timeline
    .sort_values(
        [
            "repository",
            "file_path",
            "commit_timestamp",
        ]
    )
    .groupby(
        [
            "repository",
            "file_path",
        ],
        dropna=False,
    )
    .tail(1)
    .sort_values(
        "cumulative_drift_pressure",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_repository_snapshot = (
    final_repository_timeline
    .sort_values(
        [
            "repository",
            "commit_timestamp",
        ]
    )
    .groupby(
        "repository",
        dropna=False,
    )
    .tail(1)
    .sort_values(
        "cumulative_drift_pressure",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    final_repository_snapshot[
        [
            "repository",
            "commit_timestamp",
            "commit_drift_score",
            "cumulative_drift_pressure",
            "cumulative_drift_band",
            "high_change_count",
            "critical_change_count",
        ]
    ]
)

,repository,commit_timestamp,commit_drift_score,cumulative_drift_pressure,cumulative_drift_band,high_change_count,critical_change_count
0,ansible_examples,2020-07-19 23:39:16+00:00,22.239052,48.677325,concerning,0,0
1,terraform_eks_blueprints,2026-07-07 16:04:37+00:00,20.036832,13.548165,stable,0,0


In [35]:
raw_model_results_path = (
    TABLES_DIR
    / "final_test_raw_model_results.csv"
)

raw_model_results.to_csv(
    raw_model_results_path,
    index=False,
)

variant_results_path = (
    TABLES_DIR
    / "final_test_hybrid_variant_results.csv"
)

final_variant_results.to_csv(
    variant_results_path,
    index=False,
)

per_class_results_path = (
    TABLES_DIR
    / "final_test_safety_hybrid_per_class.csv"
)

final_per_class_metrics.to_csv(
    per_class_results_path,
    index=False,
)

score_metrics_path = (
    TABLES_DIR
    / "final_test_drift_score_metrics.csv"
)

pd.DataFrame(
    [
        final_score_metrics
    ]
).to_csv(
    score_metrics_path,
    index=False,
)

score_distribution_path = (
    TABLES_DIR
    / "final_test_score_distribution.csv"
)

final_score_distribution.to_csv(
    score_distribution_path,
    index=False,
)

final_predictions_path = (
    PREDICTIONS_DIR
    / "final_test_predictions_and_scores.csv.gz"
)

final_hybrid_data.to_csv(
    final_predictions_path,
    index=False,
    compression="gzip",
)

final_commit_scores_path = (
    TABLES_DIR
    / "final_test_commit_scores.csv.gz"
)

final_commit_scores.to_csv(
    final_commit_scores_path,
    index=False,
    compression="gzip",
)

final_file_timeline_path = (
    TIMELINES_DIR
    / "final_test_file_drift_timeline.csv.gz"
)

final_file_timeline.to_csv(
    final_file_timeline_path,
    index=False,
    compression="gzip",
)

final_repository_timeline_path = (
    TIMELINES_DIR
    / "final_test_repository_drift_timeline.csv.gz"
)

final_repository_timeline.to_csv(
    final_repository_timeline_path,
    index=False,
    compression="gzip",
)

final_repository_snapshot_path = (
    TABLES_DIR
    / "final_test_repository_snapshot.csv"
)

final_repository_snapshot.to_csv(
    final_repository_snapshot_path,
    index=False,
)

critical_fn_path = (
    ERRORS_DIR
    / "final_test_critical_false_negatives.csv.gz"
)

final_critical_false_negatives.to_csv(
    critical_fn_path,
    index=False,
    compression="gzip",
)

high_critical_fn_path = (
    ERRORS_DIR
    / "final_test_high_critical_false_negatives.csv.gz"
)

final_high_critical_false_negatives.to_csv(
    high_critical_fn_path,
    index=False,
    compression="gzip",
)

high_critical_fp_path = (
    ERRORS_DIR
    / "final_test_high_critical_false_positives.csv.gz"
)

final_high_critical_false_positives.to_csv(
    high_critical_fp_path,
    index=False,
    compression="gzip",
)

print("Final predictions :", final_predictions_path)
print("Raw model metrics :", raw_model_results_path)
print("Hybrid metrics    :", variant_results_path)
print("Commit scores     :", final_commit_scores_path)

Final predictions : C:\Users\Lenovo\Desktop\DriftGuard\outputs\final_evaluation\predictions\final_test_predictions_and_scores.csv.gz
Raw model metrics : C:\Users\Lenovo\Desktop\DriftGuard\outputs\final_evaluation\tables\final_test_raw_model_results.csv
Hybrid metrics    : C:\Users\Lenovo\Desktop\DriftGuard\outputs\final_evaluation\tables\final_test_hybrid_variant_results.csv
Commit scores     : C:\Users\Lenovo\Desktop\DriftGuard\outputs\final_evaluation\tables\final_test_commit_scores.csv.gz


In [36]:
post_access_final_test_sha256 = (
    calculate_file_sha256(
        FINAL_TEST_PATH
    )
)

if (
    post_access_final_test_sha256
    != expected_final_test_sha256
):
    raise ValueError(
        "The final-test SHA-256 changed during "
        "evaluation."
    )

final_test_access_manifest = {
    "accessed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "EVALUATED_ONCE_WITH_FROZEN_PIPELINE",

    "file_path":
        str(
            FINAL_TEST_PATH.relative_to(
                PROJECT_ROOT
            )
        ),

    "expected_sha256":
        expected_final_test_sha256,

    "pre_access_sha256":
        pre_access_final_test_sha256,

    "post_access_sha256":
        post_access_final_test_sha256,

    "records":
        int(
            len(
                final_test_data
            )
        ),

    "labeled_records":
        int(
            final_test_labeled_mask.sum()
        ),

    "repositories":
        sorted(
            test_repositories
        ),

    "frozen_primary_model":
        PRIMARY_MODEL_NAME,

    "frozen_production_variant":
        PRODUCTION_VARIANT,

    "selection_changes_after_test":
        False,

    "warning": (
        "The final-test results must not be used "
        "to retune weights, thresholds, rules, "
        "bands, or model selection."
    ),
}

final_test_access_manifest_path = (
    MANIFESTS_DIR
    / "final_test_access_manifest.json"
)

with final_test_access_manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_test_access_manifest,
        file,
        indent=2,
    )

print(
    "Final-test access manifest:",
    final_test_access_manifest_path,
)

Final-test access manifest: C:\Users\Lenovo\Desktop\DriftGuard\outputs\final_evaluation\manifests\final_test_access_manifest.json


In [37]:
def collect_notebook_nodes(
    notebook_path,
):
    notebook = nbformat.read(
        notebook_path,
        as_version=4,
    )

    ordered_nodes = []
    function_nodes = {}
    assignment_nodes = {}

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        try:
            syntax_tree = ast.parse(
                cell.source
            )
        except SyntaxError:
            continue

        for node in syntax_tree.body:
            ordered_nodes.append(
                node
            )

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            ):
                function_nodes[
                    node.name
                ] = node

            elif isinstance(
                node,
                ast.Assign,
            ):
                for target in node.targets:
                    if isinstance(
                        target,
                        ast.Name,
                    ):
                        assignment_nodes[
                            target.id
                        ] = node

            elif isinstance(
                node,
                ast.AnnAssign,
            ):
                if isinstance(
                    node.target,
                    ast.Name,
                ):
                    assignment_nodes[
                        node.target.id
                    ] = node

    return {
        "ordered_nodes":
            ordered_nodes,

        "function_nodes":
            function_nodes,

        "assignment_nodes":
            assignment_nodes,
    }


def find_referenced_names(
    node,
):
    return {
        child.id
        for child in ast.walk(
            node
        )
        if isinstance(
            child,
            ast.Name,
        )
        and isinstance(
            child.ctx,
            ast.Load,
        )
    }


def build_dependency_module_source(
    notebook_path,
    entry_function_names,
):
    node_collection = (
        collect_notebook_nodes(
            notebook_path
        )
    )

    function_nodes = node_collection[
        "function_nodes"
    ]

    assignment_nodes = node_collection[
        "assignment_nodes"
    ]

    required_function_names = set()
    required_assignment_names = set()
    pending_names = list(
        entry_function_names
    )

    processed_names = set()

    while pending_names:
        current_name = pending_names.pop()

        if current_name in processed_names:
            continue

        processed_names.add(
            current_name
        )

        if current_name in function_nodes:
            required_function_names.add(
                current_name
            )

            referenced_names = (
                find_referenced_names(
                    function_nodes[
                        current_name
                    ]
                )
            )

        elif current_name in assignment_nodes:
            required_assignment_names.add(
                current_name
            )

            referenced_names = (
                find_referenced_names(
                    assignment_nodes[
                        current_name
                    ]
                )
            )

        else:
            continue

        for referenced_name in referenced_names:
            if (
                referenced_name
                in function_nodes
                or referenced_name
                in assignment_nodes
            ):
                pending_names.append(
                    referenced_name
                )

    selected_node_ids = set()

    for function_name in required_function_names:
        selected_node_ids.add(
            id(
                function_nodes[
                    function_name
                ]
            )
        )

    for assignment_name in required_assignment_names:
        selected_node_ids.add(
            id(
                assignment_nodes[
                    assignment_name
                ]
            )
        )

    source_sections = [
        "import json",
        "import math",
        "import re",
        "from collections import Counter",
        "from pathlib import Path",
        "import numpy as np",
        "import pandas as pd",
        "",
    ]

    for node in node_collection[
        "ordered_nodes"
    ]:
        if id(
            node
        ) not in selected_node_ids:
            continue

        source_sections.append(
            ast.unparse(
                node
            )
        )

        source_sections.append(
            ""
        )

    return "\n".join(
        source_sections
    )


baseline_features_source = (
    build_dependency_module_source(
        notebook_07_path,
        [
            baseline_text_builder_name,
        ],
    )
)

structured_features_source = (
    build_dependency_module_source(
        notebook_08_path,
        [
            "engineer_structured_features",
        ],
    )
)

transformer_features_source = (
    build_dependency_module_source(
        notebook_09_path,
        [
            transformer_text_builder_name,
        ],
    )
)

hybrid_rules_source = (
    build_dependency_module_source(
        notebook_11_path,
        [
            "apply_deterministic_rules_to_row",
            "build_hybrid_decision",
            "adjust_scores_to_decisions",
        ],
    )
)

scoring_source = (
    build_dependency_module_source(
        notebook_12_path,
        [
            "normalized_entropy",
            "assign_drift_band",
            "operation_is_removal",
            "operation_is_addition",
            "aggregate_event_scores",
            "calculate_decayed_pressure",
            "build_score_explanation",
        ],
    )
)

module_sources = {
    "baseline_features.py":
        baseline_features_source,

    "structured_features.py":
        structured_features_source,

    "transformer_features.py":
        transformer_features_source,

    "hybrid_rules.py":
        hybrid_rules_source,

    "scoring.py":
        scoring_source,
}

for module_filename, module_source in (
    module_sources.items()
):
    module_path = (
        DEPLOYMENT_PACKAGE_DIR
        / module_filename
    )

    module_path.write_text(
        module_source,
        encoding="utf-8",
    )

    compile(
        module_source,
        str(
            module_path
        ),
        "exec",
    )

    print(
        "Created:",
        module_path,
    )

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\baseline_features.py
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\structured_features.py
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\transformer_features.py
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\hybrid_rules.py
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\scoring.py


In [39]:
if DEPLOYMENT_MODELS_DIR.exists():
    shutil.rmtree(
        DEPLOYMENT_MODELS_DIR
    )

DEPLOYMENT_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    TEXT_MODEL_PATH,
    DEPLOYMENT_MODELS_DIR
    / "text_baseline_model.joblib",
)

shutil.copy2(
    STRUCTURED_MODEL_PATH,
    DEPLOYMENT_MODELS_DIR
    / "structured_model.joblib",
)

shutil.copytree(
    TRANSFORMER_MODEL_PATH,
    DEPLOYMENT_MODELS_DIR
    / "transformer_model",
)

if DEPLOYMENT_CONFIGS_DIR.exists():
    shutil.rmtree(
        DEPLOYMENT_CONFIGS_DIR
    )

DEPLOYMENT_CONFIGS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

configuration_files_to_copy = [
    HYBRID_SETTINGS_PATH,
    SCORING_SETTINGS_PATH,
    STRUCTURED_SETTINGS_PATH,
    PRIMARY_SELECTION_MANIFEST_PATH,
    HYBRID_MANIFEST_PATH,
    SCORING_MANIFEST_PATH,
]

for source_path in (
    configuration_files_to_copy
):
    shutil.copy2(
        source_path,
        DEPLOYMENT_CONFIGS_DIR
        / source_path.name,
    )

deployment_runtime_manifest = {
    "bundle_version": "1.0.0",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "class_order":
        CLASS_ORDER,

    "baseline_text_builder":
        baseline_text_builder_name,

    "transformer_text_builder":
        transformer_text_builder_name,

    "primary_model":
        PRIMARY_MODEL_NAME,

    "production_variant":
        PRODUCTION_VARIANT,

    "model_weights":
        MODEL_WEIGHTS,

    "model_paths": {
        "text_baseline":
            "models/text_baseline_model.joblib",

        "structured":
            "models/structured_model.joblib",

        "transformer":
            "models/transformer_model",
    },

    "final_test_access_manifest":
        str(
            final_test_access_manifest_path
            .relative_to(
                PROJECT_ROOT
            )
        ),
}

deployment_runtime_manifest_path = (
    DEPLOYMENT_CONFIGS_DIR
    / "deployment_runtime_manifest.json"
)

with deployment_runtime_manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        deployment_runtime_manifest,
        file,
        indent=2,
    )

print("Frozen models and configurations copied.")

Frozen models and configurations copied.


In [40]:
engine_source = f'''
import json
import math
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from . import baseline_features
from . import structured_features
from . import transformer_features
from . import hybrid_rules
from . import scoring


BUNDLE_ROOT = Path(__file__).resolve().parent.parent
MODELS_DIR = BUNDLE_ROOT / "models"
CONFIGS_DIR = BUNDLE_ROOT / "configs"

CLASS_ORDER = {repr(CLASS_ORDER)}
RISK_RANK = {repr(RISK_RANK)}
RANK_TO_CLASS = {{
    rank: label
    for label, rank
    in RISK_RANK.items()
}}

MODEL_WEIGHTS = {repr(MODEL_WEIGHTS)}
BASELINE_TEXT_BUILDER_NAME = {repr(baseline_text_builder_name)}
TRANSFORMER_TEXT_BUILDER_NAME = {repr(transformer_text_builder_name)}

BASELINE_TEXT_BUILDER = getattr(
    baseline_features,
    BASELINE_TEXT_BUILDER_NAME,
)

TRANSFORMER_TEXT_BUILDER = getattr(
    transformer_features,
    TRANSFORMER_TEXT_BUILDER_NAME,
)


def _load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


HYBRID_SETTINGS = _load_json(
    CONFIGS_DIR
    / "hybrid_engine_settings.json"
)

SCORING_SETTINGS = _load_json(
    CONFIGS_DIR
    / "drift_scoring_settings.json"
)

STRUCTURED_SETTINGS = _load_json(
    CONFIGS_DIR
    / "structured_model_training_settings.json"
)

structured_features.STRUCTURED_SETTINGS = (
    STRUCTURED_SETTINGS
)

hybrid_rules.HYBRID_ENGINE_SETTINGS = (
    HYBRID_SETTINGS
)

hybrid_rules.CLASS_ORDER = CLASS_ORDER
hybrid_rules.RISK_RANK = RISK_RANK
hybrid_rules.RANK_TO_CLASS = RANK_TO_CLASS
hybrid_rules.MODEL_WEIGHTS = MODEL_WEIGHTS

scoring.DRIFT_SCORING_SETTINGS = (
    SCORING_SETTINGS
)

scoring.CLASS_ORDER = CLASS_ORDER
scoring.SEVERITY_ANCHORS = (
    SCORING_SETTINGS[
        "severity_anchors"
    ]
)

scoring.BAND_TO_LABEL = (
    SCORING_SETTINGS[
        "band_to_label"
    ]
)


def _unwrap_model(saved_object):
    if not isinstance(
        saved_object,
        dict,
    ):
        return saved_object

    for key in [
        "model",
        "pipeline",
        "best_model",
        "classifier",
        "estimator",
    ]:
        if key in saved_object:
            return saved_object[key]

    raise ValueError(
        "No model object was found in the "
        "saved artifact."
    )


def _classifier_classes(model):
    if hasattr(
        model,
        "classes_",
    ):
        return np.asarray(
            model.classes_
        ).astype(str)

    if hasattr(
        model,
        "named_steps",
    ):
        for step in reversed(
            list(
                model.named_steps.values()
            )
        ):
            if hasattr(
                step,
                "classes_",
            ):
                return np.asarray(
                    step.classes_
                ).astype(str)

    raise AttributeError(
        "Classifier classes were not found."
    )


def _softmax(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    shifted = (
        values
        - values.max(
            axis=1,
            keepdims=True,
        )
    )

    exponentials = np.exp(
        shifted
    )

    return (
        exponentials
        / exponentials.sum(
            axis=1,
            keepdims=True,
        )
    )


def _align_matrix(
    matrix,
    source_classes,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    aligned = np.zeros(
        (
            matrix.shape[0],
            len(CLASS_ORDER),
        ),
        dtype=np.float64,
    )

    source_classes = [
        str(value).strip().lower()
        for value in source_classes
    ]

    for source_index, class_name in enumerate(
        source_classes
    ):
        if class_name not in CLASS_ORDER:
            continue

        aligned[
            :,
            CLASS_ORDER.index(
                class_name
            ),
        ] = matrix[
            :,
            source_index,
        ]

    return aligned


def _normalize_probabilities(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    matrix = np.clip(
        matrix,
        0.0,
        None,
    )

    zero_rows = (
        matrix.sum(
            axis=1
        )
        <= 0
    )

    if zero_rows.any():
        matrix[
            zero_rows
        ] = (
            1.0
            / len(CLASS_ORDER)
        )

    return (
        matrix
        / matrix.sum(
            axis=1,
            keepdims=True,
        )
    )


class DriftGuardEngine:
    def __init__(
        self,
        device=None,
        transformer_batch_size=8,
    ):
        self.device = torch.device(
            device
            if device is not None
            else (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )
        )

        self.transformer_batch_size = (
            transformer_batch_size
        )

        self.text_model = _unwrap_model(
            joblib.load(
                MODELS_DIR
                / "text_baseline_model.joblib"
            )
        )

        self.structured_model = _unwrap_model(
            joblib.load(
                MODELS_DIR
                / "structured_model.joblib"
            )
        )

        transformer_path = (
            MODELS_DIR
            / "transformer_model"
        )

        self.tokenizer = (
            AutoTokenizer.from_pretrained(
                transformer_path
            )
        )

        self.transformer_model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                transformer_path
            )
            .to(
                self.device
            )
        )

        self.transformer_model.eval()

    @staticmethod
    def _prepare_dataframe(
        changes,
    ):
        dataframe = pd.DataFrame(
            changes
        ).copy()

        required_columns = [
            "field_path",
            "old_value",
            "new_value",
            "configuration_type",
            "parser_mode",
            "operation",
            "file_path",
            "commit_message",
        ]

        for column in required_columns:
            if column not in dataframe.columns:
                dataframe[column] = ""

        if "repository" not in dataframe.columns:
            dataframe["repository"] = "runtime"

        if "commit_hash" not in dataframe.columns:
            dataframe["commit_hash"] = "runtime"

        if "diff_id" not in dataframe.columns:
            dataframe["diff_id"] = [
                f"runtime_{{index}}"
                for index in range(
                    len(dataframe)
                )
            ]

        return dataframe.reset_index(
            drop=True
        )

    def _run_text_model(
        self,
        dataframe,
    ):
        documents = pd.Series(
            BASELINE_TEXT_BUILDER(
                dataframe.copy()
            )
        ).fillna("").astype(str)

        predictions = self.text_model.predict(
            documents
        )

        classes = _classifier_classes(
            self.text_model
        )

        if hasattr(
            self.text_model,
            "decision_function",
        ):
            raw_scores = (
                self.text_model
                .decision_function(
                    documents
                )
            )

            aligned_scores = _align_matrix(
                raw_scores,
                classes,
            )

            probabilities = _softmax(
                aligned_scores
            )

        else:
            raw_probabilities = (
                self.text_model.predict_proba(
                    documents
                )
            )

            probabilities = _align_matrix(
                raw_probabilities,
                classes,
            )

        return (
            np.asarray(
                predictions
            ).astype(str),

            _normalize_probabilities(
                probabilities
            ),
        )

    def _run_structured_model(
        self,
        dataframe,
    ):
        features = (
            structured_features
            .engineer_structured_features(
                dataframe.copy()
            )
        )

        predictions = (
            self.structured_model.predict(
                features
            )
        )

        classes = _classifier_classes(
            self.structured_model
        )

        if hasattr(
            self.structured_model,
            "predict_proba",
        ):
            raw_probabilities = (
                self.structured_model
                .predict_proba(
                    features
                )
            )

            probabilities = _align_matrix(
                raw_probabilities,
                classes,
            )

        else:
            raw_scores = (
                self.structured_model
                .decision_function(
                    features
                )
            )

            probabilities = _softmax(
                _align_matrix(
                    raw_scores,
                    classes,
                )
            )

        return (
            np.asarray(
                predictions
            ).astype(str),

            _normalize_probabilities(
                probabilities
            ),
        )

    def _run_transformer(
        self,
        dataframe,
    ):
        documents = pd.Series(
            TRANSFORMER_TEXT_BUILDER(
                dataframe.copy()
            )
        ).fillna("").astype(str).tolist()

        batches = []

        for start in range(
            0,
            len(documents),
            self.transformer_batch_size,
        ):
            encoded = self.tokenizer(
                documents[
                    start:
                    start
                    + self.transformer_batch_size
                ],
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            )

            encoded = {{
                key: value.to(
                    self.device
                )
                for key, value
                in encoded.items()
            }}

            with torch.no_grad():
                output = self.transformer_model(
                    **encoded
                )

                probabilities = (
                    torch.softmax(
                        output.logits.float(),
                        dim=1,
                    )
                    .cpu()
                    .numpy()
                )

            batches.append(
                probabilities
            )

        probabilities = (
            _normalize_probabilities(
                np.vstack(
                    batches
                )
            )
        )

        predicted_ids = probabilities.argmax(
            axis=1
        )

        predictions = np.array(
            [
                CLASS_ORDER[
                    class_index
                ]
                for class_index
                in predicted_ids
            ]
        )

        return (
            predictions,
            probabilities,
        )

    def predict_changes(
        self,
        changes,
    ):
        dataframe = self._prepare_dataframe(
            changes
        )

        if dataframe.empty:
            return {{
                "results": [],
                "commit_summary": None,
            }}

        model_outputs = {{}}

        (
            model_outputs[
                "text_baseline"
            ],
            text_probabilities,
        ) = self._run_text_model(
            dataframe
        )

        (
            model_outputs[
                "structured"
            ],
            structured_probabilities,
        ) = self._run_structured_model(
            dataframe
        )

        (
            model_outputs[
                "transformer"
            ],
            transformer_probabilities,
        ) = self._run_transformer(
            dataframe
        )

        probabilities_by_model = {{
            "structured":
                structured_probabilities,

            "transformer":
                transformer_probabilities,

            "text_baseline":
                text_probabilities,
        }}

        result = dataframe.copy()

        for model_name in [
            "structured",
            "transformer",
            "text_baseline",
        ]:
            result[
                f"{{model_name}}_prediction"
            ] = model_outputs[
                model_name
            ]

            result[
                f"{{model_name}}_confidence"
            ] = probabilities_by_model[
                model_name
            ].max(
                axis=1
            )

            for class_index, class_name in enumerate(
                CLASS_ORDER
            ):
                result[
                    f"{{model_name}}_score_{{class_name}}"
                ] = probabilities_by_model[
                    model_name
                ][
                    :,
                    class_index,
                ]

        rule_results = result.apply(
            hybrid_rules
            .apply_deterministic_rules_to_row,
            axis=1,
            result_type="expand",
        ).rename(
            columns={{
                "matched_rule_ids":
                    "deterministic_rule_ids",
            }}
        )

        result = pd.concat(
            [
                result.reset_index(
                    drop=True
                ),
                rule_results.reset_index(
                    drop=True
                ),
            ],
            axis=1,
        )

        ensemble = np.zeros(
            (
                len(result),
                len(CLASS_ORDER),
            ),
            dtype=np.float64,
        )

        for model_name, model_weight in (
            MODEL_WEIGHTS.items()
        ):
            ensemble += (
                float(
                    model_weight
                )
                * probabilities_by_model[
                    model_name
                ]
            )

        ensemble = _normalize_probabilities(
            ensemble
        )

        ensemble_ids = ensemble.argmax(
            axis=1
        )

        result[
            "weighted_ensemble_prediction"
        ] = [
            CLASS_ORDER[
                class_index
            ]
            for class_index
            in ensemble_ids
        ]

        result[
            "weighted_ensemble_confidence"
        ] = ensemble.max(
            axis=1
        )

        for class_index, class_name in enumerate(
            CLASS_ORDER
        ):
            result[
                f"weighted_ensemble_score_{{class_name}}"
            ] = ensemble[
                :,
                class_index,
            ]

        prediction_columns = [
            "structured_prediction",
            "transformer_prediction",
            "text_baseline_prediction",
        ]

        prediction_frame = (
            result[
                prediction_columns
            ]
            .fillna("benign")
            .astype(str)
            .apply(
                lambda column:
                column.str.strip().str.lower()
            )
        )

        rank_frame = prediction_frame.replace(
            RISK_RANK
        )

        result[
            "model_unique_prediction_count"
        ] = prediction_frame.nunique(
            axis=1
        )

        result[
            "model_three_way_disagreement"
        ] = (
            result[
                "model_unique_prediction_count"
            ]
            == 3
        )

        result[
            "model_minimum_risk_rank"
        ] = rank_frame.min(
            axis=1
        )

        result[
            "model_maximum_risk_rank"
        ] = rank_frame.max(
            axis=1
        )

        result[
            "model_risk_spread"
        ] = (
            result[
                "model_maximum_risk_rank"
            ]
            - result[
                "model_minimum_risk_rank"
            ]
        )

        result[
            "high_critical_model_votes"
        ] = prediction_frame.isin(
            [
                "high",
                "critical",
            ]
        ).sum(
            axis=1
        )

        result[
            "critical_model_votes"
        ] = prediction_frame.eq(
            "critical"
        ).sum(
            axis=1
        )

        result[
            "ensemble_high_critical_probability"
        ] = (
            result[
                "weighted_ensemble_score_high"
            ]
            + result[
                "weighted_ensemble_score_critical"
            ]
        )

        hybrid_decisions = result.apply(
            hybrid_rules
            .build_hybrid_decision,
            axis=1,
            result_type="expand",
        )

        result = pd.concat(
            [
                result,
                hybrid_decisions,
            ],
            axis=1,
        )

        hybrid_probabilities = (
            hybrid_rules
            .adjust_scores_to_decisions(
                ensemble,

                result[
                    "safety_hybrid_prediction"
                ].tolist(),

                boost=(
                    HYBRID_SETTINGS[
                        "hybrid_score_boost"
                    ]
                ),
            )
        )

        hybrid_probabilities = (
            _normalize_probabilities(
                hybrid_probabilities
            )
        )

        result[
            "safety_hybrid_confidence"
        ] = hybrid_probabilities.max(
            axis=1
        )

        for class_index, class_name in enumerate(
            CLASS_ORDER
        ):
            result[
                f"safety_hybrid_score_{{class_name}}"
            ] = hybrid_probabilities[
                :,
                class_index,
            ]

        severity_anchor_vector = np.array(
            [
                SCORING_SETTINGS[
                    "severity_anchors"
                ][
                    class_name
                ]
                for class_name in CLASS_ORDER
            ],
            dtype=np.float64,
        )

        expected_risk = (
            hybrid_probabilities
            @ severity_anchor_vector
        )

        decision_anchor = (
            result[
                "safety_hybrid_prediction"
            ]
            .map(
                SCORING_SETTINGS[
                    "severity_anchors"
                ]
            )
            .to_numpy(
                dtype=np.float64
            )
        )

        model_component = (
            SCORING_SETTINGS[
                "score_weights"
            ][
                "decision_anchor"
            ]
            * decision_anchor

            + SCORING_SETTINGS[
                "score_weights"
            ][
                "expected_probability_risk"
            ]
            * expected_risk
        )

        rule_anchor = (
            result[
                "rule_severity"
            ]
            .map(
                SCORING_SETTINGS[
                    "severity_anchors"
                ]
            )
            .fillna(0.0)
            .to_numpy(
                dtype=np.float64
            )
        )

        rule_confidence = (
            result[
                "rule_confidence"
            ]
            .fillna(0.0)
            .clip(
                lower=0.0,
                upper=1.0,
            )
            .to_numpy(
                dtype=np.float64
            )
        )

        rule_floor = np.where(
            result[
                "rule_match_count"
            ].fillna(0).gt(0),

            rule_anchor
            * (
                0.65
                + 0.35
                * rule_confidence
            ),

            0.0,
        )

        critical_votes = result[
            "critical_model_votes"
        ].to_numpy(
            dtype=np.float64
        )

        high_critical_votes = result[
            "high_critical_model_votes"
        ].to_numpy(
            dtype=np.float64
        )

        supported_probability = result[
            "ensemble_high_critical_probability"
        ].to_numpy(
            dtype=np.float64
        )

        consensus_bonus = np.select(
            [
                critical_votes >= 2,

                high_critical_votes >= 2,

                (
                    critical_votes >= 1
                )
                & (
                    supported_probability
                    >= SCORING_SETTINGS[
                        "supported_critical_probability"
                    ]
                ),
            ],
            [
                SCORING_SETTINGS[
                    "consensus_bonus"
                ][
                    "two_critical_votes"
                ],

                SCORING_SETTINGS[
                    "consensus_bonus"
                ][
                    "two_high_critical_votes"
                ],

                SCORING_SETTINGS[
                    "consensus_bonus"
                ][
                    "supported_single_critical_vote"
                ],
            ],
            default=0.0,
        )

        change_scores = np.clip(
            np.maximum(
                model_component,
                rule_floor,
            )
            + consensus_bonus,
            0.0,
            100.0,
        )

        entropy = scoring.normalized_entropy(
            hybrid_probabilities
        )

        result[
            "expected_probability_risk"
        ] = expected_risk

        result[
            "model_risk_component"
        ] = model_component

        result[
            "rule_risk_floor"
        ] = rule_floor

        result[
            "consensus_score_bonus"
        ] = consensus_bonus

        result[
            "change_risk_score"
        ] = change_scores

        result[
            "uncertainty_score"
        ] = entropy * 100.0

        result[
            "drift_band"
        ] = result[
            "change_risk_score"
        ].apply(
            scoring.assign_drift_band
        )

        output_columns = [
            "diff_id",
            "repository",
            "commit_hash",
            "file_path",
            "field_path",
            "old_value",
            "new_value",
            "operation",
            "configuration_type",
            "safety_hybrid_prediction",
            "safety_hybrid_confidence",
            "change_risk_score",
            "uncertainty_score",
            "drift_band",
            "deterministic_rule_ids",
            "decision_source",
            "decision_reason",
        ]

        commit_aggregate = (
            scoring.aggregate_event_scores(
                result[
                    "change_risk_score"
                ].to_numpy()
            )
        )

        commit_summary = {{
            "changed_fields":
                int(
                    len(result)
                ),

            "changed_files":
                int(
                    result[
                        "file_path"
                    ].nunique()
                ),

            "maximum_change_score":
                commit_aggregate[
                    "maximum_score"
                ],

            "top_three_mean_score":
                commit_aggregate[
                    "top_three_mean_score"
                ],

            "risk_mass_score":
                commit_aggregate[
                    "risk_mass_score"
                ],

            "commit_drift_score":
                commit_aggregate[
                    "event_score"
                ],

            "commit_drift_band":
                scoring.assign_drift_band(
                    commit_aggregate[
                        "event_score"
                    ]
                ),
        }}

        records = json.loads(
            result[
                output_columns
            ].to_json(
                orient="records"
            )
        )

        return {{
            "results": records,
            "commit_summary": commit_summary,
        }}
'''

engine_path = (
    DEPLOYMENT_PACKAGE_DIR
    / "engine.py"
)

engine_path.write_text(
    textwrap.dedent(
        engine_source
    ),
    encoding="utf-8",
)

compile(
    engine_path.read_text(
        encoding="utf-8"
    ),
    str(
        engine_path
    ),
    "exec",
)

print("Created:", engine_path)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\engine.py


In [41]:
schemas_source = '''
from typing import Any

from pydantic import BaseModel, Field


class ConfigurationChange(BaseModel):
    diff_id: str | None = None
    repository: str = "runtime"
    commit_hash: str = "runtime"
    field_path: str = ""
    old_value: Any = ""
    new_value: Any = ""
    configuration_type: str = ""
    parser_mode: str = "structured"
    operation: str = "modified"
    file_path: str = ""
    commit_message: str = ""


class PredictionRequest(BaseModel):
    changes: list[ConfigurationChange] = Field(
        min_length=1
    )


class PredictionResponse(BaseModel):
    results: list[dict]
    commit_summary: dict | None
'''

app_source = '''
from fastapi import FastAPI

from .engine import DriftGuardEngine
from .schemas import (
    PredictionRequest,
    PredictionResponse,
)


app = FastAPI(
    title="DriftGuard Inference API",
    version="1.0.0",
    description=(
        "Configuration-drift risk classification "
        "and cumulative commit scoring."
    ),
)

engine = DriftGuardEngine()


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "service": "driftguard",
    }


@app.post(
    "/predict",
    response_model=PredictionResponse,
)
def predict(
    request: PredictionRequest,
):
    changes = [
        change.model_dump()
        for change in request.changes
    ]

    return engine.predict_changes(
        changes
    )
'''

init_source = '''
from .engine import DriftGuardEngine

__all__ = ["DriftGuardEngine"]
'''

(
    DEPLOYMENT_PACKAGE_DIR
    / "schemas.py"
).write_text(
    textwrap.dedent(
        schemas_source
    ),
    encoding="utf-8",
)

(
    DEPLOYMENT_PACKAGE_DIR
    / "app.py"
).write_text(
    textwrap.dedent(
        app_source
    ),
    encoding="utf-8",
)

(
    DEPLOYMENT_PACKAGE_DIR
    / "__init__.py"
).write_text(
    textwrap.dedent(
        init_source
    ),
    encoding="utf-8",
)

for module_filename in [
    "schemas.py",
    "app.py",
    "__init__.py",
]:
    module_path = (
        DEPLOYMENT_PACKAGE_DIR
        / module_filename
    )

    compile(
        module_path.read_text(
            encoding="utf-8"
        ),
        str(
            module_path
        ),
        "exec",
    )

    print("Created:", module_path)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\schemas.py
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\app.py
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\__init__.py


In [42]:
example_request = {
    "changes": [
        {
            "diff_id":
                "example_001",

            "repository":
                "example-service",

            "commit_hash":
                "abc123",

            "field_path":
                "spec.template.spec.containers[0].securityContext.runAsNonRoot",

            "old_value":
                "true",

            "new_value":
                "false",

            "configuration_type":
                "yaml",

            "parser_mode":
                "structured",

            "operation":
                "modified",

            "file_path":
                "kubernetes/deployment.yaml",

            "commit_message":
                "update container configuration",
        },

        {
            "diff_id":
                "example_002",

            "repository":
                "example-service",

            "commit_hash":
                "abc123",

            "field_path":
                "spec.resources.limits.memory",

            "old_value":
                "512Mi",

            "new_value":
                "",

            "configuration_type":
                "yaml",

            "parser_mode":
                "structured",

            "operation":
                "deleted",

            "file_path":
                "kubernetes/deployment.yaml",

            "commit_message":
                "update container configuration",
        },
    ]
}

example_request_path = (
    DEPLOYMENT_EXAMPLES_DIR
    / "prediction_request.json"
)

with example_request_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        example_request,
        file,
        indent=2,
    )

example_response = {
    "results": [
        {
            "diff_id":
                "example_001",

            "safety_hybrid_prediction":
                "high",

            "change_risk_score":
                75.0,

            "drift_band":
                "high",

            "deterministic_rule_ids":
                "",

            "decision_source":
                "weighted_model_ensemble",
        }
    ],

    "commit_summary": {
        "changed_fields": 2,
        "changed_files": 1,
        "commit_drift_score": 75.0,
        "commit_drift_band": "high",
    },
}

example_response_path = (
    DEPLOYMENT_EXAMPLES_DIR
    / "prediction_response_example.json"
)

with example_response_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        example_response,
        file,
        indent=2,
    )

print("Created:", example_request_path)
print("Created:", example_response_path)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\examples\prediction_request.json
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\examples\prediction_response_example.json


In [43]:
def installed_version(
    distribution_name,
):
    try:
        return importlib.metadata.version(
            distribution_name
        )
    except Exception:
        return None


required_packages = {
    "numpy":
        installed_version(
            "numpy"
        ),

    "pandas":
        installed_version(
            "pandas"
        ),

    "scikit-learn":
        installed_version(
            "scikit-learn"
        ),

    "joblib":
        installed_version(
            "joblib"
        ),

    "torch":
        installed_version(
            "torch"
        ),

    "transformers":
        installed_version(
            "transformers"
        ),

    "tokenizers":
        installed_version(
            "tokenizers"
        ),

    "safetensors":
        installed_version(
            "safetensors"
        ),
}

requirement_lines = []

for package_name, package_version in (
    required_packages.items()
):
    if package_version:
        requirement_lines.append(
            f"{package_name}=={package_version}"
        )
    else:
        requirement_lines.append(
            package_name
        )

requirement_lines.extend(
    [
        "fastapi>=0.110",
        "uvicorn[standard]>=0.27",
        "pydantic>=2.0",
    ]
)

requirements_path = (
    DEPLOYMENT_ROOT
    / "requirements.txt"
)

requirements_path.write_text(
    "\n".join(
        requirement_lines
    )
    + "\n",
    encoding="utf-8",
)

print(
    requirements_path.read_text(
        encoding="utf-8"
    )
)

numpy==2.4.6
pandas==3.0.3
scikit-learn==1.9.0
joblib==1.5.3
torch==2.11.0+cu128
transformers==5.14.1
tokenizers==0.22.2
safetensors==0.8.0
fastapi>=0.110
uvicorn[standard]>=0.27
pydantic>=2.0



In [44]:
smoke_test_source = '''
from driftguard_inference.engine import (
    DriftGuardEngine,
)


def test_driftguard_smoke():
    engine = DriftGuardEngine(
        device="cpu",
        transformer_batch_size=1,
    )

    response = engine.predict_changes(
        [
            {
                "diff_id": "smoke_001",
                "repository": "smoke-test",
                "commit_hash": "smoke123",
                "field_path": "tls.enabled",
                "old_value": "true",
                "new_value": "false",
                "configuration_type": "yaml",
                "parser_mode": "structured",
                "operation": "modified",
                "file_path": "config.yaml",
                "commit_message": "disable tls",
            }
        ]
    )

    assert len(
        response["results"]
    ) == 1

    result = response[
        "results"
    ][0]

    assert result[
        "safety_hybrid_prediction"
    ] in {
        "benign",
        "low",
        "medium",
        "high",
        "critical",
    }

    assert (
        0.0
        <= result[
            "change_risk_score"
        ]
        <= 100.0
    )

    assert response[
        "commit_summary"
    ] is not None
'''

smoke_test_path = (
    DEPLOYMENT_TESTS_DIR
    / "test_smoke.py"
)

smoke_test_path.write_text(
    textwrap.dedent(
        smoke_test_source
    ),
    encoding="utf-8",
)

print("Created:", smoke_test_path)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\tests\test_smoke.py


In [46]:
model_card_text = f"""
# DriftGuard Model Card

## System

DriftGuard detects potentially risky configuration drift in Git-managed
infrastructure and application configuration files.

The exported system combines:

- A character and word TF-IDF linear classifier
- A structured ExtraTrees classifier
- A CodeBERTa-small Transformer classifier
- Eight deterministic security rules
- A safety-oriented hybrid decision engine
- A 0–100 drift scoring and cumulative-pressure engine

## Frozen production configuration

- Primary learned model: `{PRIMARY_MODEL_NAME}`
- Production decision engine: `{PRODUCTION_VARIANT}`
- Model weights: `{MODEL_WEIGHTS}`
- Class order: `{CLASS_ORDER}`

## Final test protocol

The final test was repository-disjoint and remained sealed until all model,
rule, hybrid, and drift-scoring settings had been frozen.

Final test repositories:

`{sorted(test_repositories)}`

Final-test SHA-256:

`{expected_final_test_sha256}`

## Final production-hybrid metrics

- Accuracy: {production_final_row['accuracy']:.6f}
- Balanced accuracy: {production_final_row['balanced_accuracy']:.6f}
- Macro F1: {production_final_row['macro_f1']:.6f}
- Weighted F1: {production_final_row['weighted_f1']:.6f}
- Macro PR-AUC: {production_final_row['macro_pr_auc']:.6f}
- Critical precision: {production_final_row['critical_precision']:.6f}
- Critical recall: {production_final_row['critical_recall']:.6f}
- High/critical precision: {production_final_row['high_critical_precision']:.6f}
- High/critical recall: {production_final_row['high_critical_recall']:.6f}

## Important limitations

1. Evaluation labels are weak security labels rather than fully independent
   human-verified ground truth.
2. Deterministic-rule metrics partly measure consistency with the weak-label
   rules.
3. Repository and configuration-type distribution shifts can significantly
   reduce generalization.
4. High/critical predictions should be reviewed by a security or platform
   engineer.
5. The cumulative drift-pressure value represents recent risk pressure. It is
   not proof that every previous issue remains active.
6. The system does not execute configuration files or verify whether a
   deployment is currently exploitable.
7. Secrets and configuration values should be handled according to the
   organization's data-retention and access-control requirements.

## Intended use

- Pull-request security screening
- Infrastructure-as-code review
- Configuration-change triage
- Drift hotspot identification
- Commit-level risk prioritization

## Out-of-scope use

- Autonomous blocking without review
- Exploitability confirmation
- Compliance certification
- Replacement for secret scanning, SAST, DAST, policy-as-code, or manual review
"""

model_card_path = (
    DEPLOYMENT_DOCS_DIR
    / "MODEL_CARD.md"
)

model_card_path.write_text(
    textwrap.dedent(
        model_card_text
    ).strip()
    + "\n",
    encoding="utf-8",
)

print("Created:", model_card_path)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\docs\MODEL_CARD.md


In [49]:
# Corrected Cell 40 — Generate integration guide and README

integration_guide_lines = [
    "# DriftGuard Integration Guide",
    "",
    "## Install",
    "",
    "Open a terminal inside the backend bundle directory:",
    "",
    "    pip install -r requirements.txt",
    "",
    "## Run the API",
    "",
    "    uvicorn driftguard_inference.app:app --host 0.0.0.0 --port 8000",
    "",
    "## Health check",
    "",
    "Open:",
    "",
    "    http://localhost:8000/health",
    "",
    "Or run:",
    "",
    "    curl http://localhost:8000/health",
    "",
    "## Prediction request",
    "",
    "Windows Command Prompt:",
    "",
    "    curl -X POST http://localhost:8000/predict ^",
    '      -H "Content-Type: application/json" ^',
    "      --data @examples/prediction_request.json",
    "",
    "PowerShell:",
    "",
    "    Invoke-RestMethod `",
    '      -Uri "http://localhost:8000/predict" `',
    "      -Method Post `",
    '      -ContentType "application/json" `',
    '      -InFile "examples/prediction_request.json"',
    "",
    "## Direct Python use",
    "",
    "    from driftguard_inference import DriftGuardEngine",
    "",
    "    engine = DriftGuardEngine()",
    "",
    "    response = engine.predict_changes(",
    "        [",
    "            {",
    '                "field_path": "tls.enabled",',
    '                "old_value": "true",',
    '                "new_value": "false",',
    '                "configuration_type": "yaml",',
    '                "parser_mode": "structured",',
    '                "operation": "modified",',
    '                "file_path": "config.yaml",',
    '                "commit_message": "disable tls",',
    "            }",
    "        ]",
    "    )",
    "",
    "    print(response)",
    "",
    "## Input fields",
    "",
    "Operational fields:",
    "",
    "- `field_path`",
    "- `old_value`",
    "- `new_value`",
    "- `configuration_type`",
    "- `parser_mode`",
    "- `operation`",
    "- `file_path`",
    "- `commit_message`",
    "",
    "Optional identifiers:",
    "",
    "- `diff_id`",
    "- `repository`",
    "- `commit_hash`",
    "",
    "## Output interpretation",
    "",
    "- `safety_hybrid_prediction`: final categorical risk class",
    "- `safety_hybrid_confidence`: confidence of the hybrid prediction",
    "- `change_risk_score`: field-level risk score from 0 to 100",
    "- `uncertainty_score`: uncertainty in the probability distribution",
    "- `drift_band`: stable, watch, concerning, high, or critical",
    "- `deterministic_rule_ids`: matched security rules",
    "- `decision_source`: source of the final decision",
    "- `decision_reason`: reason for model or rule intervention",
    "- `commit_summary`: aggregate risk for submitted changes",
    "",
    "## Deployment recommendations",
    "",
    "- Protect the endpoint using authentication.",
    "- Do not record raw secrets in application logs.",
    "- Introduce request-size limits.",
    "- Send high and critical predictions for human review.",
    "- Store model and configuration hashes with every deployment.",
    "- Do not alter frozen thresholds using final-test results.",
]

integration_guide_text = "\n".join(
    integration_guide_lines
)


readme_lines = [
    "# DriftGuard Backend Bundle",
    "",
    "This directory contains the frozen DriftGuard inference system.",
    "",
    "## Contents",
    "",
    "- `driftguard_inference/` — inference engine and FastAPI application",
    "- `models/` — text, structured, and Transformer models",
    "- `configs/` — frozen hybrid and scoring configurations",
    "- `examples/` — API request and response examples",
    "- `tests/` — deployment smoke test",
    "- `docs/` — model card and integration guide",
    "- `requirements.txt` — Python dependencies",
    "",
    "See `docs/INTEGRATION_GUIDE.md` for execution instructions.",
]

readme_text = "\n".join(
    readme_lines
)


integration_guide_path = (
    DEPLOYMENT_DOCS_DIR
    / "INTEGRATION_GUIDE.md"
)

integration_guide_path.write_text(
    integration_guide_text.strip()
    + "\n",
    encoding="utf-8",
)


readme_path = (
    DEPLOYMENT_ROOT
    / "README.md"
)

readme_path.write_text(
    readme_text.strip()
    + "\n",
    encoding="utf-8",
)


print(
    "Created:",
    integration_guide_path,
)

print(
    "Created:",
    readme_path,
)

print(
    "\nCell 40 completed successfully."
)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\docs\INTEGRATION_GUIDE.md
Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\README.md

Cell 40 completed successfully.


In [52]:



## Cell 41 — Save final metrics inside the bundle


def convert_json_value(value):
    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(value)

    if isinstance(
        value,
        (
            int,
            np.integer,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            float,
            np.floating,
        ),
    ):
        return float(value)

    return value


final_metrics_payload = {
    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "final_test": {
        "records":
            int(
                len(
                    final_test_data
                )
            ),

        "labeled_records":
            int(
                final_test_labeled_mask.sum()
            ),

        "repositories":
            sorted(
                test_repositories
            ),

        "sha256":
            expected_final_test_sha256,
    },

    "raw_models":
        raw_model_results.to_dict(
            orient="records"
        ),

    "hybrid_variants":
        final_variant_results.to_dict(
            orient="records"
        ),

    "production_variant":
        PRODUCTION_VARIANT,

    "production_metrics": {
        key: convert_json_value(value)
        for key, value
        in production_final_row.to_dict().items()
    },

    "score_metrics": {
        key: convert_json_value(value)
        for key, value
        in final_score_metrics.items()
    },

    "critical_false_negatives":
        int(
            len(
                final_critical_false_negatives
            )
        ),

    "high_critical_false_negatives":
        int(
            len(
                final_high_critical_false_negatives
            )
        ),

    "high_critical_false_positives":
        int(
            len(
                final_high_critical_false_positives
            )
        ),
}

bundle_metrics_path = (
    DEPLOYMENT_DOCS_DIR
    / "FINAL_TEST_METRICS.json"
)

with bundle_metrics_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_metrics_payload,
        file,
        indent=2,
    )

print("Created:", bundle_metrics_path)

Created: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\docs\FINAL_TEST_METRICS.json


In [54]:
# Patch exported transformer_features.py

transformer_features_path = (
    DEPLOYMENT_PACKAGE_DIR
    / "transformer_features.py"
)

source_text = transformer_features_path.read_text(
    encoding="utf-8"
)

missing_constants_block = """
# Runtime-safe Transformer settings
TRAIN_BATCH_SIZE = 2
EVALUATION_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 8
"""

if "TRAIN_BATCH_SIZE = 2" not in source_text:
    marker = "RANDOM_SEED = 42"

    if marker not in source_text:
        raise RuntimeError(
            "Could not find RANDOM_SEED assignment in "
            "transformer_features.py"
        )

    source_text = source_text.replace(
        marker,
        marker + "\n" + missing_constants_block,
        1,
    )

    transformer_features_path.write_text(
        source_text,
        encoding="utf-8",
    )

print(
    "Patched:",
    transformer_features_path,
)

print(
    "Transformer runtime constants added successfully."
)

Patched: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\transformer_features.py
Transformer runtime constants added successfully.


In [55]:
print("=" * 72)
print("RUNNING DEPLOYMENT-BUNDLE SMOKE TEST")
print("=" * 72)

deployment_parent = str(
    DEPLOYMENT_ROOT
)

if deployment_parent not in sys.path:
    sys.path.insert(
        0,
        deployment_parent,
    )

# Remove previously cached package versions.
for module_name in list(
    sys.modules.keys()
):
    if (
        module_name
        == "driftguard_inference"
        or module_name.startswith(
            "driftguard_inference."
        )
    ):
        del sys.modules[
            module_name
        ]

deployment_engine_module = (
    importlib.import_module(
        "driftguard_inference.engine"
    )
)

DeploymentEngine = (
    deployment_engine_module
    .DriftGuardEngine
)

deployment_engine = DeploymentEngine(
    device=(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    ),
    transformer_batch_size=1,
)

smoke_response = (
    deployment_engine.predict_changes(
        example_request[
            "changes"
        ][
            :1
        ]
    )
)

print(
    json.dumps(
        smoke_response,
        indent=2,
    )
)

if len(
    smoke_response[
        "results"
    ]
) != 1:
    raise RuntimeError(
        "Deployment smoke inference failed: "
        "incorrect number of output records."
    )

smoke_result = smoke_response[
    "results"
][0]

if (
    smoke_result[
        "safety_hybrid_prediction"
    ]
    not in CLASS_ORDER
):
    raise RuntimeError(
        "Deployment returned an invalid class."
    )

if not (
    0.0
    <= smoke_result[
        "change_risk_score"
    ]
    <= 100.0
):
    raise RuntimeError(
        "Deployment returned an invalid risk score."
    )

if (
    smoke_response[
        "commit_summary"
    ]
    is None
):
    raise RuntimeError(
        "Deployment did not generate a "
        "commit-level summary."
    )

print("\nDeployment smoke inference: PASSED")

RUNNING DEPLOYMENT-BUNDLE SMOKE TEST


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{
  "results": [
    {
      "diff_id": "example_001",
      "repository": "example-service",
      "commit_hash": "abc123",
      "file_path": "kubernetes/deployment.yaml",
      "field_path": "spec.template.spec.containers[0].securityContext.runAsNonRoot",
      "old_value": "true",
      "new_value": "false",
      "operation": "modified",
      "configuration_type": "yaml",
      "safety_hybrid_prediction": "high",
      "safety_hybrid_confidence": 0.7466200488,
      "change_risk_score": 75.6031903976,
      "uncertainty_score": 51.365481843,
      "drift_band": "high",
      "deterministic_rule_ids": "",
      "decision_source": "weighted_model_ensemble",
      "decision_reason": "Weighted ensemble prediction: high."
    }
  ],
  "commit_summary": {
    "changed_fields": 1,
    "changed_files": 1,
    "maximum_change_score": 75.60319039762813,
    "top_three_mean_score": 75.60319039762813,
    "risk_mass_score": 17.222009569920495,
    "commit_drift_score": 63.9269542320866,
    

In [56]:
deployment_zip_path = (
    PROJECT_ROOT
    / "deployment"
    / "driftguard_backend_bundle.zip"
)

if deployment_zip_path.exists():
    deployment_zip_path.unlink()

with zipfile.ZipFile(
    deployment_zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:

    for source_path in (
        DEPLOYMENT_ROOT.rglob("*")
    ):
        if not source_path.is_file():
            continue

        relative_path = (
            source_path.relative_to(
                DEPLOYMENT_ROOT.parent
            )
        )

        zip_file.write(
            source_path,
            arcname=str(
                relative_path
            ),
        )

print(
    "Deployment ZIP:",
    deployment_zip_path,
)

print(
    "ZIP size:",
    f"{deployment_zip_path.stat().st_size / (1024 ** 2):.2f} MB",
)

Deployment ZIP: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle.zip
ZIP size: 323.04 MB


In [57]:
final_system_manifest = {
    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "system_name":
        "DriftGuard",

    "system_version":
        "1.0.0",

    "primary_model":
        PRIMARY_MODEL_NAME,

    "production_variant":
        PRODUCTION_VARIANT,

    "class_order":
        CLASS_ORDER,

    "model_weights":
        MODEL_WEIGHTS,

    "final_test": {
        "status":
            final_test_access_manifest[
                "status"
            ],

        "records":
            int(
                len(
                    final_test_data
                )
            ),

        "labeled_records":
            int(
                final_test_labeled_mask.sum()
            ),

        "repositories":
            sorted(
                test_repositories
            ),

        "sha256":
            expected_final_test_sha256,
    },

    "production_metrics": {
        key: convert_json_value(value)
        for key, value
        in production_final_row.to_dict().items()
    },

    "score_metrics": {
        key: convert_json_value(value)
        for key, value
        in final_score_metrics.items()
    },

    "deployment": {
        "bundle_directory":
            str(
                DEPLOYMENT_ROOT.relative_to(
                    PROJECT_ROOT
                )
            ),

        "bundle_zip":
            str(
                deployment_zip_path.relative_to(
                    PROJECT_ROOT
                )
            ),

        "api_module":
            "driftguard_inference.app:app",

        "engine_class":
            (
                "driftguard_inference.engine."
                "DriftGuardEngine"
            ),
    },

    "output_files": {
        "predictions":
            str(
                final_predictions_path.relative_to(
                    PROJECT_ROOT
                )
            ),

        "raw_model_results":
            str(
                raw_model_results_path.relative_to(
                    PROJECT_ROOT
                )
            ),

        "hybrid_results":
            str(
                variant_results_path.relative_to(
                    PROJECT_ROOT
                )
            ),

        "commit_scores":
            str(
                final_commit_scores_path.relative_to(
                    PROJECT_ROOT
                )
            ),

        "repository_timeline":
            str(
                final_repository_timeline_path.relative_to(
                    PROJECT_ROOT
                )
            ),
    },

    "post_test_changes": {
        "model_selection_changed":
            False,

        "model_weights_changed":
            False,

        "rule_definitions_changed":
            False,

        "score_thresholds_changed":
            False,

        "drift_bands_changed":
            False,
    },
}

final_system_manifest_path = (
    MANIFESTS_DIR
    / "final_system_manifest.json"
)

with final_system_manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_system_manifest,
        file,
        indent=2,
    )

shutil.copy2(
    final_system_manifest_path,
    DEPLOYMENT_CONFIGS_DIR
    / "final_system_manifest.json",
)

print(
    "Final system manifest:",
    final_system_manifest_path,
)

Final system manifest: C:\Users\Lenovo\Desktop\DriftGuard\outputs\final_evaluation\manifests\final_system_manifest.json


In [58]:
deployment_required_files = [
    DEPLOYMENT_PACKAGE_DIR
    / "__init__.py",

    DEPLOYMENT_PACKAGE_DIR
    / "engine.py",

    DEPLOYMENT_PACKAGE_DIR
    / "app.py",

    DEPLOYMENT_PACKAGE_DIR
    / "schemas.py",

    DEPLOYMENT_PACKAGE_DIR
    / "baseline_features.py",

    DEPLOYMENT_PACKAGE_DIR
    / "structured_features.py",

    DEPLOYMENT_PACKAGE_DIR
    / "transformer_features.py",

    DEPLOYMENT_PACKAGE_DIR
    / "hybrid_rules.py",

    DEPLOYMENT_PACKAGE_DIR
    / "scoring.py",

    DEPLOYMENT_ROOT
    / "requirements.txt",

    DEPLOYMENT_ROOT
    / "README.md",

    DEPLOYMENT_DOCS_DIR
    / "MODEL_CARD.md",

    DEPLOYMENT_DOCS_DIR
    / "INTEGRATION_GUIDE.md",

    DEPLOYMENT_DOCS_DIR
    / "FINAL_TEST_METRICS.json",

    DEPLOYMENT_TESTS_DIR
    / "test_smoke.py",
]

final_integrity_checks = {
    "Final-test row count is correct":
        len(
            final_test_data
        )
        == 68_749,

    "Final-test pre-access hash matches":
        pre_access_final_test_sha256
        == expected_final_test_sha256,

    "Final-test post-access hash matches":
        post_access_final_test_sha256
        == expected_final_test_sha256,

    "Final-test repositories are disjoint":
        len(
            repository_overlap
        )
        == 0,

    "Final-test commits are disjoint":
        len(
            commit_overlap
        )
        == 0,

    "Final-test signatures are disjoint":
        len(
            signature_overlap
        )
        == 0,

    "All three models produced predictions":
        set(
            final_model_outputs.keys()
        )
        == {
            "structured",
            "transformer",
            "text_baseline",
        },

    "All prediction counts match final test":
        all(
            len(
                output[
                    "predicted_labels"
                ]
            )
            == len(
                final_test_data
            )
            for output
            in final_model_outputs.values()
        ),

    "All production predictions are valid":
        final_hybrid_data[
            "safety_hybrid_prediction"
        ].isin(
            CLASS_ORDER
        ).all(),

    "All probability rows sum to one":
        np.allclose(
            safety_hybrid_probabilities.sum(
                axis=1
            ),
            1.0,
            atol=1e-8,
        ),

    "All change scores are valid":
        final_hybrid_data[
            "change_risk_score"
        ].between(
            0.0,
            100.0,
            inclusive="both",
        ).all(),

    "All drift bands are valid":
        final_hybrid_data[
            "drift_band"
        ].isin(
            [
                "stable",
                "watch",
                "concerning",
                "high",
                "critical",
            ]
        ).all(),

    "Primary model was not changed":
        PRIMARY_MODEL_NAME
        == primary_selection_manifest[
            "selected_primary_model"
        ],

    "Production variant was not changed":
        PRODUCTION_VARIANT
        == hybrid_manifest[
            "production_variant"
        ],

    "No post-test tuning is recorded":
        not any(
            final_system_manifest[
                "post_test_changes"
            ].values()
        ),

    "Deployment files exist":
        all(
            path.exists()
            for path
            in deployment_required_files
        ),

    "Deployment smoke inference passed":
        len(
            smoke_response[
                "results"
            ]
        )
        == 1,

    "Deployment ZIP exists":
        deployment_zip_path.exists(),

    "Final system manifest exists":
        final_system_manifest_path.exists(),
}

print("Final system integrity checks:\n")

for check_name, passed in (
    final_integrity_checks.items()
):
    print(
        f"{'PASSED' if passed else 'FAILED':<8}"
        f" | {check_name}"
    )

failed_final_checks = [
    check_name
    for check_name, passed
    in final_integrity_checks.items()
    if not bool(
        passed
    )
]

print(
    "\nFailed integrity checks:",
    len(
        failed_final_checks
    ),
)

if failed_final_checks:
    print("\nFailed checks:")

    for check_number, check_name in enumerate(
        failed_final_checks,
        start=1,
    ):
        print(
            f"{check_number}. {check_name}"
        )
else:
    print(
        "All final system integrity checks passed."
    )

Final system integrity checks:

PASSED   | Final-test row count is correct
PASSED   | Final-test pre-access hash matches
PASSED   | Final-test post-access hash matches
PASSED   | Final-test repositories are disjoint
PASSED   | Final-test commits are disjoint
PASSED   | Final-test signatures are disjoint
PASSED   | All three models produced predictions
PASSED   | All prediction counts match final test
PASSED   | All production predictions are valid
PASSED   | All probability rows sum to one
PASSED   | All change scores are valid
PASSED   | All drift bands are valid
PASSED   | Primary model was not changed
PASSED   | Production variant was not changed
PASSED   | No post-test tuning is recorded
PASSED   | Deployment files exist
PASSED   | Deployment smoke inference passed
PASSED   | Deployment ZIP exists
PASSED   | Final system manifest exists

Failed integrity checks: 0
All final system integrity checks passed.


In [59]:
print("=" * 72)
print("NOTEBOOK 13 — REQUIRED RESULTS")
print("=" * 72)

print("\n1. FINAL-TEST INTEGRITY")
print("-" * 72)

print(
    "File:",
    FINAL_TEST_PATH.name,
)

print(
    "Records:",
    f"{len(final_test_data):,}",
)

print(
    "Labeled records:",
    f"{final_test_labeled_mask.sum():,}",
)

print(
    "Repositories:",
    sorted(
        test_repositories
    ),
)

print(
    "Repository overlap:",
    len(
        repository_overlap
    ),
)

print(
    "Commit overlap:",
    len(
        commit_overlap
    ),
)

print(
    "Signature overlap:",
    len(
        signature_overlap
    ),
)

print(
    "Pre-access SHA-256:",
    pre_access_final_test_sha256,
)

print(
    "Post-access SHA-256:",
    post_access_final_test_sha256,
)


print("\n2. RAW MODEL FINAL-TEST RESULTS")
print("-" * 72)

display(
    raw_model_results
)


print("\n3. FROZEN HYBRID FINAL-TEST RESULTS")
print("-" * 72)

display(
    final_variant_results
)


print("\n4. PRODUCTION SAFETY-HYBRID")
print("-" * 72)

print(
    "Production variant        :",
    PRODUCTION_VARIANT,
)

print(
    "Accuracy                  :",
    f"{production_final_row['accuracy']:.6f}",
)

print(
    "Balanced accuracy         :",
    f"{production_final_row['balanced_accuracy']:.6f}",
)

print(
    "Macro F1                  :",
    f"{production_final_row['macro_f1']:.6f}",
)

print(
    "Macro PR-AUC              :",
    f"{production_final_row['macro_pr_auc']:.6f}",
)

print(
    "Critical precision        :",
    f"{production_final_row['critical_precision']:.6f}",
)

print(
    "Critical recall           :",
    f"{production_final_row['critical_recall']:.6f}",
)

print(
    "High/critical precision   :",
    f"{production_final_row['high_critical_precision']:.6f}",
)

print(
    "High/critical recall      :",
    f"{production_final_row['high_critical_recall']:.6f}",
)


print("\n5. PRODUCTION PER-CLASS RESULTS")
print("-" * 72)

display(
    final_per_class_metrics
)


print("\n6. FINAL SAFETY ERRORS")
print("-" * 72)

print(
    "Critical false negatives     :",
    f"{len(final_critical_false_negatives):,}",
)

print(
    "High/critical false negatives:",
    f"{len(final_high_critical_false_negatives):,}",
)

print(
    "High/critical false positives:",
    f"{len(final_high_critical_false_positives):,}",
)


print("\n7. FINAL DRIFT SCORE RESULTS")
print("-" * 72)

print(
    "Mean change score:",
    f"{final_hybrid_data['change_risk_score'].mean():.6f}",
)

print(
    "Median change score:",
    f"{final_hybrid_data['change_risk_score'].median():.6f}",
)

print(
    "High/critical changes:",
    f"{final_hybrid_data['change_risk_score'].ge(60).sum():,}",
)

print(
    "Critical changes:",
    f"{final_hybrid_data['change_risk_score'].ge(80).sum():,}",
)

display(
    final_score_distribution
)


print("\n8. FINAL SCORE EVALUATION")
print("-" * 72)

for metric_name, metric_value in (
    final_score_metrics.items()
):
    if isinstance(
        metric_value,
        float,
    ):
        print(
            f"{metric_name:<32}: "
            f"{metric_value:.6f}"
        )
    else:
        print(
            f"{metric_name:<32}: "
            f"{metric_value}"
        )


print("\n9. FINAL COMMIT AND REPOSITORY DRIFT")
print("-" * 72)

print(
    "Scored commits:",
    f"{len(final_commit_scores):,}",
)

print(
    "High/critical commits:",
    f"{final_commit_scores['commit_drift_score'].ge(60).sum():,}",
)

print(
    "Critical commits:",
    f"{final_commit_scores['commit_drift_score'].ge(80).sum():,}",
)

display(
    final_repository_snapshot[
        [
            "repository",
            "commit_timestamp",
            "commit_drift_score",
            "cumulative_drift_pressure",
            "cumulative_drift_band",
            "high_change_count",
            "critical_change_count",
        ]
    ]
)


print("\n10. DEPLOYMENT EXPORT")
print("-" * 72)

print(
    "Bundle directory:",
    DEPLOYMENT_ROOT,
)

print(
    "Bundle ZIP:",
    deployment_zip_path,
)

print(
    "Bundle ZIP size:",
    f"{deployment_zip_path.stat().st_size / (1024 ** 2):.2f} MB",
)

print(
    "Inference engine:",
    DEPLOYMENT_PACKAGE_DIR
    / "engine.py",
)

print(
    "FastAPI app:",
    DEPLOYMENT_PACKAGE_DIR
    / "app.py",
)

print(
    "Model card:",
    model_card_path,
)

print(
    "Integration guide:",
    integration_guide_path,
)


print("\n11. FINAL-TEST STATUS")
print("-" * 72)

print(
    "Status:",
    final_test_access_manifest[
        "status"
    ],
)

print(
    "Post-test model reselection:",
    False,
)

print(
    "Post-test threshold tuning:",
    False,
)


print("\n12. FAILED INTEGRITY CHECKS")
print("-" * 72)

if failed_final_checks:
    for index, check_name in enumerate(
        failed_final_checks,
        start=1,
    ):
        print(
            f"{index}. {check_name}"
        )
else:
    print(
        "No failed integrity checks."
    )

    print(
        "The final evaluation and deployment "
        "export passed all checks."
    )

print("\n" + "=" * 72)
print("REQUIRED RESULTS GENERATED")
print("=" * 72)

NOTEBOOK 13 — REQUIRED RESULTS

1. FINAL-TEST INTEGRITY
------------------------------------------------------------------------
File: test_final_evaluation_only.csv.gz
Records: 68,749
Labeled records: 61,899
Repositories: ['ansible_examples', 'terraform_eks_blueprints']
Repository overlap: 0
Commit overlap: 0
Signature overlap: 0
Pre-access SHA-256: cff0f85db740a195366fb81cbad26ba34859f40fe12bf8695b263d1cdedb9b67
Post-access SHA-256: cff0f85db740a195366fb81cbad26ba34859f40fe12bf8695b263d1cdedb9b67

2. RAW MODEL FINAL-TEST RESULTS
------------------------------------------------------------------------


,reporting_rank,model,records,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_pr_auc,log_loss,critical_precision,critical_recall,critical_f1,high_critical_precision,high_critical_recall,high_critical_f1,inference_seconds
0,1,text_baseline,61899,0.591415,0.516079,0.389371,0.623980,0.449601,1.331315,0.028450,0.062500,0.039101,0.056801,0.228,0.090945,194.984574
1,2,structured,61899,0.515727,0.419490,0.338477,0.548732,0.356153,2.940574,0.053145,0.153125,0.078905,0.075088,0.170,0.104167,10.777380
2,3,transformer,61899,0.437568,0.459366,0.310426,0.461793,0.406193,6.271581,0.045025,0.253125,0.076451,0.059574,0.364,0.102391,340.275178



3. FROZEN HYBRID FINAL-TEST RESULTS
------------------------------------------------------------------------


,variant,records,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_pr_auc,log_loss,critical_precision,critical_recall,critical_f1,high_critical_precision,high_critical_recall,high_critical_f1,production_variant
0,primary_structured,61899,0.515727,0.419490,0.338477,0.548732,0.356179,2.940574,0.053145,0.153125,0.078905,0.075088,0.170,0.104167,False
1,weighted_ensemble,61899,0.445645,0.473511,0.343073,0.467687,0.461875,1.569128,0.044798,0.193750,0.072770,0.067936,0.314,0.111704,False
2,rules_plus_ensemble,61899,0.424062,0.619747,0.273441,0.456302,0.464210,1.761253,0.051136,0.590625,0.094124,0.063747,0.588,0.115023,False
3,safety_hybrid,61899,0.423028,0.623216,0.269263,0.455773,0.463909,1.760715,0.050821,0.600000,0.093704,0.061425,0.600,0.111441,True



4. PRODUCTION SAFETY-HYBRID
------------------------------------------------------------------------
Production variant        : safety_hybrid
Accuracy                  : 0.423028
Balanced accuracy         : 0.623216
Macro F1                  : 0.269263
Macro PR-AUC              : 0.463909
Critical precision        : 0.050821
Critical recall           : 0.600000
High/critical precision   : 0.061425
High/critical recall      : 0.600000

5. PRODUCTION PER-CLASS RESULTS
------------------------------------------------------------------------


...


6. FINAL SAFETY ERRORS
------------------------------------------------------------------------
Critical false negatives     : 128
High/critical false negatives: 200
High/critical false positives: 4,584

7. FINAL DRIFT SCORE RESULTS
------------------------------------------------------------------------
Mean change score: 24.846493
Median change score: 24.909147
High/critical changes: 5,084
Critical changes: 3,822


,drift_band,records,percentage
0,stable,24064,35.002691
1,watch,39005,56.735371
2,concerning,596,0.866922
3,high,1262,1.835663
4,critical,3822,5.559354



8. FINAL SCORE EVALUATION
------------------------------------------------------------------------
labeled_records                 : 61899
accuracy                        : 0.420734
balanced_accuracy               : 0.569791
macro_f1                        : 0.253899
weighted_f1                     : 0.453516
severity_rank_correlation       : 0.273894
mean_absolute_anchor_error      : 17.274685
critical_precision              : 0.050235
critical_recall                 : 0.600000
critical_f1                     : 0.092709
high_critical_precision         : 0.061425
high_critical_recall            : 0.600000
high_critical_f1                : 0.111441

9. FINAL COMMIT AND REPOSITORY DRIFT
------------------------------------------------------------------------
Scored commits: 1,252
High/critical commits: 362
Critical commits: 237


,repository,commit_timestamp,commit_drift_score,cumulative_drift_pressure,cumulative_drift_band,high_change_count,critical_change_count
0,ansible_examples,2020-07-19 23:39:16+00:00,22.239052,48.677325,concerning,0,0
1,terraform_eks_blueprints,2026-07-07 16:04:37+00:00,20.036832,13.548165,stable,0,0



10. DEPLOYMENT EXPORT
------------------------------------------------------------------------
Bundle directory: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle
Bundle ZIP: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle.zip
Bundle ZIP size: 323.04 MB
Inference engine: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\engine.py
FastAPI app: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\driftguard_inference\app.py
Model card: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\docs\MODEL_CARD.md
Integration guide: C:\Users\Lenovo\Desktop\DriftGuard\deployment\driftguard_backend_bundle\docs\INTEGRATION_GUIDE.md

11. FINAL-TEST STATUS
------------------------------------------------------------------------
Status: EVALUATED_ONCE_WITH_FROZEN_PIPELINE
Post-test model reselection: False
Post-test threshold tuning: False

12. FAILED INTEGRITY CHECKS
-------